# Dose Whisperer — Notebook 3: Model Setup, Base vs QLoRA, Comparison

> ## ⚠️ Research prototype — not a medical device
> Dose Whisperer produces **prototype safety flags** (`SAFE` / `WARNING` / `HIGH`). These are **not medical diagnoses, not treatment advice, and not a replacement for clinical judgment**. MIMIC-III Demo prescriptions are real examples but are **not assumed to be clinically safe ground truth**. All error labels in this project come from **synthetic perturbations**.


## 1. Install, imports and version checks
Set `INSTALL_DEPS = True` on a fresh machine (Colab/Kaggle/cloud VM). A GPU runtime is required for the LLM sections.

In [ ]:
# ===== 1. INSTALL / IMPORTS / VERSION CHECKS =====
import subprocess, sys
INSTALL_DEPS = False  # set True ONLY on a fresh machine; restart the kernel afterwards if torch/transformers were upgraded
if INSTALL_DEPS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
        "transformers>=4.50", "peft>=0.14", "bitsandbytes>=0.45", "accelerate>=1.0",
        "huggingface_hub", "sacrebleu", "rouge-score", "nltk", "bert-score",
        "scikit-learn", "matplotlib", "pandas", "numpy", "tqdm", "pyarrow"])

import os, re, gc, json, math, time, random, shutil, inspect, platform, traceback, warnings, getpass
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
from importlib import metadata as _md

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display, Markdown, Image
import matplotlib
import matplotlib.pyplot as plt

import torch
import transformers
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer, EarlyStoppingCallback, set_seed as hf_set_seed)
from transformers.trainer_utils import get_last_checkpoint
try:
    from transformers import AutoModelForImageTextToText      # MedGemma / Gemma-3 (multimodal) loading class
except ImportError:
    AutoModelForImageTextToText = None
import peft
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from huggingface_hub import login as hf_login, hf_hub_download

warnings.filterwarnings("ignore", message=".*use_cache=True.*")

def _ver(pkg):
    try: return _md.version(pkg)
    except Exception: return "not installed"

print("Python", platform.python_version(), "|", platform.platform())
print("Library versions:")
for p in ["torch", "transformers", "peft", "bitsandbytes", "accelerate", "huggingface_hub",
          "sacrebleu", "rouge-score", "nltk", "bert-score", "pandas", "numpy", "matplotlib"]:
    print(f"  {p:16s} {_ver(p)}")

HAS_CUDA = torch.cuda.is_available()
if HAS_CUDA:
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    BF16_OK = torch.cuda.is_bf16_supported()
    print(f"GPU: {GPU_NAME} | VRAM {GPU_VRAM_GB:.1f} GB | bf16 supported: {BF16_OK}")
else:
    GPU_NAME, GPU_VRAM_GB, BF16_OK = "none", 0.0, False
    print("⚠️  No CUDA GPU detected. 4-bit (bitsandbytes) loading, training and generation need a GPU. "
          "Data/rules/metrics sections will still run.")

# NLTK WordNet is needed for METEOR (download needs internet; failure just disables METEOR).
try:
    import nltk
    for _r in ["wordnet", "omw-1.4"]:
        nltk.download(_r, quiet=True)
except Exception as e:
    print("NLTK WordNet download failed (METEOR may be NaN):", repr(e))

## 2. Configuration and hardware profile
**Everything configurable lives in this one cell.** Fill the four paths and (if needed) the column map. One switch (`PROFILE`) changes batch sizes, evidence caps, subset sizes, optimizer and which models can be QLoRA-trained — the *logic* is identical for both profiles.

In [ ]:
# ===== 2. CONFIG & HARDWARE PROFILE (edit this cell only) =====
PROFILE      = "a100_80gb"     # "a100_80gb" (primary) | "low_vram_6gb" (optional)
QUICK_MODE   = True           # smoke test: tiny subsets + 1 epoch (use with USE_DEMO_DATA=True for a full dry run)
USE_DEMO_DATA = False          # True => generate a tiny SYNTHETIC demo dataset (pipeline smoke test ONLY, not clinical data)
SEED         = 42
# Separate output folders per mode (…_quick / …_full): smoke-test runs must never share cached predictions/adapters with the real run.
# (The old un-suffixed folder from the first smoke test is deliberately never reused.)
OUT_DIR      = Path("./dose_whisperer_nb3_outputs_" + ("quick" if QUICK_MODE else "full") + ("_demo" if USE_DEMO_DATA else ""))

# ---- input data (FILL THESE; the notebook validates them in section 4) ----
TRAIN_PATH    = "/home/user9/SOURABH_S/data/splits/train.csv"
VAL_PATH      = "/home/user9/SOURABH_S/data/splits/val.csv"
TEST_PATH     = "/home/user9/SOURABH_S/data/splits/test.csv"
EVIDENCE_PATH = "/home/user9/SOURABH_S/data/splits/evidence.csv"       # only needed if the example files do not already carry evidence columns

# ---- column mapping: your column name for each field (None = not available) ----
COLS = dict(
    example_id="example_id",            # optional: auto-generated if missing
    group_id="original_id",             # id of the ORIGINAL prescription (used for the leakage check)
    drug="drug", dose="dose", unit="unit", frequency="frequency",
    conditions="conditions",            # list, JSON list string, or "a; b | c"
    error_type="error_type",            # optional but recommended (stratification + error analysis)
    y_dose="dose_status", y_unit="unit_status", y_freq="frequency_status",
    y_cond="condition_risk", y_comb="combined_risk",
    ev_dosage="evidence_dosage", ev_warnings="evidence_warnings", ev_contra="evidence_contraindications",
    # ---- optional extras (set to a column name to use them) ----
    ref_explanation=None,               # gold/reference explanation text
    ref_matched_warning=None,           # gold matched warning text
    ref_evidence=None,                  # gold evidence id/quote
    range_min=None, range_max=None, range_unit=None, max_daily=None,   # precomputed label dose range (same unit as range_unit)
)
# Columns of EVIDENCE_PATH (one or more rows per drug; rows for the same drug are concatenated)
EVID_COLS = dict(drug="drug", dosage="dosage", warnings="warnings", contraindications="contraindications")
CONDITION_SEPARATORS = r"[;|\n]"
COMBINED_MODE = "both_required"          # "both_required" | "overall_max"   (see section 0, assumption 4)

# ---- models ----
PHI_VARIANT = "microsoft/Phi-3-mini-4k-instruct"      # or "microsoft/Phi-3.5-mini-instruct"
ENABLE = dict(medgemma=True, phi=True, qwen=False, llama=False)   # qwen / llama = optional extras (off by default)
MODEL_REGISTRY = {
    "medgemma": dict(hf_id="google/medgemma-4b-it", gated=True, family="gemma3", needs_token_type_ids=True,
                     train_attn="eager",   # Gemma-3 recommends eager attention for training
                     lora_targets=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)"),
    "phi":      dict(hf_id=PHI_VARIANT, gated=False, family="phi3", needs_token_type_ids=False, train_attn="sdpa",
                     lora_targets=["qkv_proj", "o_proj", "gate_up_proj", "down_proj"]),
    "qwen":     dict(hf_id="Qwen/Qwen2.5-3B-Instruct", gated=False, family="qwen2", needs_token_type_ids=False, train_attn="sdpa",
                     lora_targets=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]),
    "llama":    dict(hf_id="meta-llama/Llama-3.2-3B-Instruct", gated=True, family="llama", needs_token_type_ids=False, train_attn="sdpa",
                     lora_targets=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]),
}
RUN_BF16_REFERENCE = True      # extra "Base bf16 + RAG" reference row (A100 profile only)
FORCE_RERUN = False            # True => ignore cached predictions/adapters (does NOT unlock a second test pass, see 11b)

# ---- hardware profiles ----
PROFILES = {
    "a100_80gb": dict(gen_bs=32, train_bs=8, grad_accum=2, grad_ckpt=False, optim="adamw_torch",
                      max_seq_len=3072, snippet_chars=350, max_snippets=8, section_chars=1800,
                      train_max=4000, val_dev_max=400, test_max=None, val_loss_max=300,
                      skip_qlora=[], attn="sdpa", compute_dtype="bfloat16"),
    "low_vram_6gb": dict(gen_bs=4, train_bs=1, grad_accum=16, grad_ckpt=True, optim="paged_adamw_8bit",
                         max_seq_len=1536, snippet_chars=220, max_snippets=5, section_chars=700,
                         train_max=1200, val_dev_max=100, test_max=200, val_loss_max=100,
                         skip_qlora=["medgemma"],     # MedGemma QLoRA skipped => inference-only on this profile
                         attn="sdpa", compute_dtype="float16"),
}
P = dict(PROFILES[PROFILE])
if PROFILE == "a100_80gb" and BF16_OK is False and HAS_CUDA:
    P["compute_dtype"] = "float16"
if PROFILE == "low_vram_6gb":
    RUN_BF16_REFERENCE = False

# ---- QLoRA hyper-parameters (conservative, fixed a-priori; validation used ONLY for early stopping + dev eval) ----
TRAIN_HP = dict(learning_rate=1e-4, epochs=2, lora_r=16, lora_alpha=32, lora_dropout=0.05,
                warmup_ratio=0.05, weight_decay=0.0, max_grad_norm=0.3, lr_scheduler="cosine",
                early_stopping_patience=2, evals_per_epoch=3)
# 512 (A100) / 384 (6 GB): Phi-3 writes long explanations; 256-320 truncated its JSON mid-sentence (=> false "invalid JSON").
# Identical in QUICK_MODE and full runs so the smoke test is representative.
MAX_NEW_TOKENS = 512 if PROFILE == "a100_80gb" else 384
DECODING = dict(do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS)   # greedy => identical for every run
PROMPT_VERSION = "dw-prompt-v1.0"

# ---- deterministic checker thresholds ----
CHK = dict(over_high=2.0,     # dose > label max by more than 2x  => HIGH (between 1x and 2x => WARNING)
           under_high=5.0,    # dose < label min by 5x or more    => HIGH (between 1x and 5x => WARNING)
           daily_high=2.0,    # daily dose > label max/day by >2x => HIGH
           unit_high=10.0,    # unit mislabel with >=10x magnitude gap => HIGH
           single_ref_lo_tol=2.0)   # when only a single reference dose was found, accept down to ref/2

# ---- quick-mode overrides ----
if QUICK_MODE:
    P.update(train_max=64, val_dev_max=24, test_max=24, val_loss_max=16)
    TRAIN_HP["epochs"] = 1
    TRAIN_HP["evals_per_epoch"] = 2

# ---- output folders ----
for sub in ["preds", "adapters", "ckpt", "figs", "logs", "tables"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

TEST_UNLOCKED = False   # flipped to True only in section 11b

def set_seed_all(seed=SEED):
    random.seed(seed); np.random.seed(seed); hf_set_seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed_all()

def now_iso(): return datetime.now(timezone.utc).isoformat(timespec="seconds")

# ---- tiny persistent state + failure log (fault tolerance) ----
STATE_PATH = OUT_DIR / "state.json"
def _load_state():
    try: return json.loads(STATE_PATH.read_text())
    except Exception: return {}
STATE = _load_state()
def mark_done(key, payload=True):
    STATE[key] = payload; STATE_PATH.write_text(json.dumps(STATE, indent=1, default=str))
def is_done(key): return (not FORCE_RERUN) and key in STATE

FAIL_LOG = OUT_DIR / "logs" / "failures.jsonl"
def log_failure(step, exc, tb):
    with open(FAIL_LOG, "a") as f:
        f.write(json.dumps(dict(time=now_iso(), step=step, error=repr(exc), traceback=tb)) + "\n")

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()

def safe_run(step, fn, *args, **kw):
    """Run one step; on failure log + continue (one broken model must not stop the others)."""
    try:
        return fn(*args, **kw)
    except Exception as e:
        tb = traceback.format_exc(); log_failure(step, e, tb)
        print(f"❌ [{step}] failed: {e!r}\n   (logged to {FAIL_LOG}; continuing)")
        return None
    finally:
        clear_gpu()

# ---- stale-cache guard: cached predictions / adapters are only valid for the settings that produced them ----
RUN_FINGERPRINT = dict(profile=PROFILE, quick_mode=QUICK_MODE, demo_data=USE_DEMO_DATA, seed=SEED, prompt_version=PROMPT_VERSION,
                       decoding=DECODING, chk=CHK, combined_mode=COMBINED_MODE, phi_variant=PHI_VARIANT, profile_settings=P,
                       train_hp=TRAIN_HP, data=[TRAIN_PATH, VAL_PATH, TEST_PATH, EVIDENCE_PATH])
_FP_PATH = OUT_DIR / "logs" / "run_fingerprint.json"
_fp_now = json.dumps(RUN_FINGERPRINT, sort_keys=True, default=str, indent=1)
if _FP_PATH.exists() and _FP_PATH.read_text() != _fp_now:
    if FORCE_RERUN:
        print("WARNING: settings differ from the cached run in OUT_DIR. FORCE_RERUN=True regenerates train/val caches, but cached TEST "
              "predictions are never regenerated - use a fresh OUT_DIR if you changed prompts, decoding or training.")
    else:
        raise RuntimeError(f"Settings differ from the run that produced the cached files in {OUT_DIR}. Cached predictions/adapters would be "
                           f"stale. Use a fresh OUT_DIR (or move the old folder away), or set FORCE_RERUN=True knowingly. "
                           f"Compare with {_FP_PATH}.")
_FP_PATH.write_text(_fp_now)

print(f"PROFILE={PROFILE} | QUICK_MODE={QUICK_MODE} | DEMO_DATA={USE_DEMO_DATA} | seed={SEED}")
print("Profile settings:", json.dumps({k: v for k, v in P.items()}, default=str))
print("Training HP:", TRAIN_HP)
print("Outputs ->", OUT_DIR.resolve())

## 3. Hugging Face login and gated-access check
The token is read from the `HF_TOKEN` environment variable, otherwise you are prompted (hidden input). **No secret is stored in the notebook.** MedGemma is gated: accept the *Health AI Developer Foundations* terms at https://huggingface.co/google/medgemma-4b-it with the same account first. Models whose access check fails are **skipped and logged** (the others still run). MedGemma is for research/developer use only.

In [ ]:
# ===== 3. HF LOGIN + GATED ACCESS CHECK =====
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
needs_token = any(ENABLE.get(k) and MODEL_REGISTRY[k]["gated"] for k in MODEL_REGISTRY)
if HF_TOKEN is None and needs_token:
    try:
        HF_TOKEN = getpass.getpass("Hugging Face token (hidden; leave empty to skip gated models): ").strip() or None
    except Exception:
        HF_TOKEN = None
if HF_TOKEN:
    hf_login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face.")
else:
    print("No HF token provided: gated models (MedGemma, Llama) will be skipped.")

def check_model_access(key):
    cfg = MODEL_REGISTRY[key]
    if cfg["gated"] and not HF_TOKEN:
        return False, "gated model but no HF token provided"
    try:
        hf_hub_download(cfg["hf_id"], "config.json", token=HF_TOKEN)   # tiny file; raises if access is denied
        return True, "ok"
    except Exception as e:
        name = type(e).__name__
        hint = (" -> accept the model terms on its Hugging Face page with this token's account." if "Gated" in name or "403" in str(e) else "")
        return False, f"{name}: {str(e)[:160]}{hint}"

ACTIVE_MODELS = []
for key, on in ENABLE.items():
    if not on: continue
    ok, msg = check_model_access(key)
    print(f"{'✅' if ok else '⛔'} {key:9s} {MODEL_REGISTRY[key]['hf_id']:42s} {msg}")
    if ok: ACTIVE_MODELS.append(key)
    else: log_failure(f"access:{key}", RuntimeError(msg), "")
if not ACTIVE_MODELS:
    print("⚠️  No model is accessible. Data/rules sections still run, but LLM sections will be skipped.")
MODEL_NAME = {k: MODEL_REGISTRY[k]["hf_id"].split("/")[-1] for k in MODEL_REGISTRY}
print("Active models:", ACTIVE_MODELS)

## 4. Data loading and validation
Loads the three split files (+ evidence), **validates paths and columns with clear messages**, normalises labels, checks that **no original prescription appears in more than one split** (leakage check), and takes deterministic, seeded subsets according to the hardware profile. The same test subset is used for every system.

In [ ]:
# ===== 4a. (optional) tiny SYNTHETIC demo data generator — pipeline smoke test only =====
# Texts below are short ILLUSTRATIVE paraphrases for exercising the code. They are NOT clinical references.
DEMO_DRUGS = {
 "acetaminophen": dict(dose=500, unit="mg", lo=500, hi=1000, maxd=4000, fmin=4, fmax=6, freqs=["q6h", "q8h"],
    dosage="Adults and children 12 years and over: 500 mg to 1000 mg every 4 to 6 hours as needed. Do not exceed 4000 mg in 24 hours.",
    warnings="Liver warning: severe liver damage may occur if the user has 3 or more alcoholic drinks every day. Use caution in patients with hepatic impairment or liver disease. Allergy alert: acetaminophen may cause severe skin reactions.",
    contra="Do not use in patients with known hypersensitivity to acetaminophen.",
    risk={"hepatic impairment": "WARNING"}),
 "ibuprofen": dict(dose=400, unit="mg", lo=200, hi=400, maxd=3200, fmin=4, fmax=6, freqs=["q6h", "q8h"],
    dosage="The recommended dose is 200 mg to 400 mg every 4 to 6 hours. Do not exceed 3200 mg per day.",
    warnings="NSAIDs cause an increased risk of serious gastrointestinal adverse events including bleeding, ulceration, and perforation. Use caution in patients with heart failure or fluid retention. Renal toxicity has been reported in patients with renal impairment.",
    contra="Ibuprofen is contraindicated in patients with active peptic ulcer disease. Contraindicated in the setting of coronary artery bypass graft surgery.",
    risk={"peptic ulcer disease": "HIGH", "heart failure": "WARNING", "renal impairment": "WARNING"}),
 "metformin": dict(dose=500, unit="mg", lo=250, hi=500, maxd=2550, fmin=2, fmax=2, freqs=["bid"],
    dosage="The usual starting dose is 500 mg twice daily. Doses may be increased to a maximum of 2550 mg per day given in divided doses.",
    warnings="Lactic acidosis risk increases with hepatic impairment and excessive alcohol intake. Monitor renal function.",
    contra="Metformin is contraindicated in patients with severe renal impairment (eGFR below 30 mL/min/1.73 m2).",
    risk={"renal impairment": "HIGH", "hepatic impairment": "WARNING"}),
 "digoxin": dict(dose=250, unit="mcg", lo=125, hi=250, maxd=250, fmin=1, fmax=1, freqs=["qd"],
    dosage="The usual maintenance dose is 125 mcg to 250 mcg once daily.",
    warnings="Use caution in patients with renal impairment because digoxin is eliminated by the kidneys. Hypokalemia increases the risk of toxicity.",
    contra="Digoxin is contraindicated in patients with ventricular fibrillation.",
    risk={"renal impairment": "WARNING"}),
 "lisinopril": dict(dose=20, unit="mg", lo=20, hi=40, maxd=40, fmin=1, fmax=1, freqs=["qd"],
    dosage="The usual initial dose is 20 mg once daily. The usual dosage range is 20 mg to 40 mg once daily.",
    warnings="Use caution in patients with renal impairment and hyperkalemia.",
    contra="Lisinopril is contraindicated in patients with a history of angioedema. Discontinue lisinopril as soon as possible when pregnancy is detected because of fetal toxicity.",
    risk={"pregnancy": "HIGH", "renal impairment": "WARNING"}),
}
_FREQ_PER_DAY = {"qd": 1, "bid": 2, "tid": 3, "qid": 4, "q8h": 3, "q6h": 4, "q4h": 6, "q3h": 8, "q2h": 12, "q1h": 24}

def make_demo_data(n_orig=150, seed=SEED):
    """Writes train/val/test/evidence CSVs under OUT_DIR/demo_data and points the *_PATH globals at them."""
    global TRAIN_PATH, VAL_PATH, TEST_PATH, EVIDENCE_PATH
    rng = random.Random(seed)
    L = {"SAFE": 0, "WARNING": 1, "HIGH": 2}; N = ["SAFE", "WARNING", "HIGH"]
    to_mg = {"mg": 1.0, "mcg": 1e-3, "g": 1e3}
    pool = ["hepatic impairment", "renal impairment", "peptic ulcer disease", "heart failure",
            "pregnancy", "hypertension", "type 2 diabetes", "asthma"]
    def st_range(v, lo, hi):
        if v > hi + 1e-9: return "WARNING" if v / hi <= 2.0 + 1e-9 else "HIGH"
        if v < lo - 1e-9: return "WARNING" if lo / v < 5.0 - 1e-9 else "HIGH"
        return "SAFE"
    def over(x, lim):  # x/lim > 1
        return "SAFE" if x <= lim + 1e-9 else ("WARNING" if x / lim <= 2.0 + 1e-9 else "HIGH")
    rows = []
    for i in range(n_orig):
        d = rng.choice(list(DEMO_DRUGS)); sp = DEMO_DRUGS[d]
        conds = rng.sample(pool, rng.choice([1, 2]))
        std_f = rng.choice(sp["freqs"])
        variants = [dict(et="none", dose=sp["dose"], unit=sp["unit"], freq=std_f)]
        for _ in range(rng.choice([1, 2, 3])):
            et = rng.choice(["dose_x2", "dose_x5", "dose_x10", "dose_/2", "dose_/10", "unit", "frequency"])
            v = dict(et=et, dose=sp["dose"], unit=sp["unit"], freq=std_f)
            if et.startswith("dose_x"): v["dose"] = sp["dose"] * float(et.split("x")[1])
            elif et.startswith("dose_/"): v["dose"] = sp["dose"] / float(et.split("/")[1])
            elif et == "unit": v["unit"] = "mcg" if sp["unit"] == "mg" else "mg"
            else: v["freq"] = rng.choice([f for f in _FREQ_PER_DAY if f != std_f])
            variants.append(v)
        crisk = max((sp["risk"].get(c, "SAFE") for c in conds), key=lambda s: L[s])
        for k, v in enumerate(variants):
            ref_u = sp["unit"]; per_day = _FREQ_PER_DAY[v["freq"]]
            if v["unit"] == ref_u:
                dose_st, unit_st, v_eff = st_range(v["dose"], sp["lo"], sp["hi"]), "SAFE", v["dose"]
            else:
                conv = v["dose"] * to_mg[v["unit"]] / to_mg[ref_u]
                if st_range(conv, sp["lo"], sp["hi"]) == "SAFE": dose_st, unit_st, v_eff = "SAFE", "SAFE", conv
                elif st_range(v["dose"], sp["lo"], sp["hi"]) == "SAFE":
                    gap = max(conv / sp["hi"], sp["lo"] / conv); dose_st, unit_st, v_eff = "SAFE", ("HIGH" if gap >= 10 else "WARNING"), v["dose"]
                else: dose_st, unit_st, v_eff = st_range(conv, sp["lo"], sp["hi"]), "SAFE", conv
            f_st = over(per_day, sp["fmax"])
            if per_day < sp["fmin"] and sp["fmin"] / per_day >= 2.0: f_st = max(f_st, "WARNING", key=lambda s: L[s])
            if dose_st == "SAFE": f_st = max(f_st, over(v_eff * per_day, sp["maxd"]), key=lambda s: L[s])
            dose_risk = max(dose_st, unit_st, f_st, key=lambda s: L[s])
            both = dose_risk != "SAFE" and crisk != "SAFE"
            comb = N[min(2, max(L[dose_risk], L[crisk]) + 1)] if both else "SAFE"
            if COMBINED_MODE == "overall_max": comb = max(dose_risk, crisk, key=lambda s: L[s])
            rows.append({COLS["example_id"]: f"d{i}_{k}", COLS["group_id"]: f"orig{i}", COLS["drug"]: d,
                         COLS["dose"]: v["dose"], COLS["unit"]: v["unit"], COLS["frequency"]: v["freq"],
                         COLS["conditions"]: "; ".join(conds), COLS["error_type"]: v["et"],
                         COLS["y_dose"]: dose_st, COLS["y_unit"]: unit_st, COLS["y_freq"]: f_st,
                         COLS["y_cond"]: crisk, COLS["y_comb"]: comb})
    df = pd.DataFrame(rows)
    gids = sorted(df[COLS["group_id"]].unique()); rng.shuffle(gids)
    n = len(gids); tr, va = set(gids[:int(.7 * n)]), set(gids[int(.7 * n):int(.85 * n)])
    d_out = OUT_DIR / "demo_data"; d_out.mkdir(exist_ok=True, parents=True)
    g = df[COLS["group_id"]]
    df[g.isin(tr)].to_csv(d_out / "train.csv", index=False)
    df[g.isin(va)].to_csv(d_out / "val.csv", index=False)
    df[~g.isin(tr | va)].to_csv(d_out / "test.csv", index=False)
    pd.DataFrame([{EVID_COLS["drug"]: k, EVID_COLS["dosage"]: v["dosage"], EVID_COLS["warnings"]: v["warnings"],
                   EVID_COLS["contraindications"]: v["contra"]} for k, v in DEMO_DRUGS.items()]).to_csv(d_out / "evidence.csv", index=False)
    TRAIN_PATH, VAL_PATH, TEST_PATH, EVIDENCE_PATH = (str(d_out / f) for f in ["train.csv", "val.csv", "test.csv", "evidence.csv"])
    print(f"⚠️  DEMO DATA written to {d_out} ({len(df)} synthetic rows). Use only to smoke-test the pipeline.")

if USE_DEMO_DATA:
    make_demo_data()

In [ ]:
# ===== 4b. LOAD + VALIDATE =====
def _unset(p): return p is None or str(p).strip() == "" or str(p).strip().startswith("<")

def read_table(path):
    p = Path(path); ext = p.suffix.lower()
    if ext == ".csv":                return pd.read_csv(p)
    if ext == ".tsv":                return pd.read_csv(p, sep="\t")
    if ext in (".jsonl", ".ndjson"): return pd.read_json(p, lines=True, dtype=False)
    if ext == ".json":               return pd.read_json(p, dtype=False)
    if ext == ".parquet":            return pd.read_parquet(p)
    raise ValueError(f"Unsupported file type '{ext}' for {p} (use csv/tsv/jsonl/json/parquet)")

def _fail(msg):
    print("\n" + "=" * 78 + "\n❌ DATA CONFIGURATION PROBLEM\n" + msg + "\n" + "=" * 78)
    raise RuntimeError(msg.splitlines()[0])

# --- paths ---
bad = [f"  {n} = {p!r}" for n, p in [("TRAIN_PATH", TRAIN_PATH), ("VAL_PATH", VAL_PATH), ("TEST_PATH", TEST_PATH)]
       if _unset(p) or not Path(p).exists()]
if bad:
    _fail("Missing/unfilled data paths (edit the config cell in section 2, or set USE_DEMO_DATA=True for a smoke test):\n" + "\n".join(bad))

RAW = {s: read_table(p) for s, p in [("train", TRAIN_PATH), ("val", VAL_PATH), ("test", TEST_PATH)]}

# --- columns ---
REQUIRED = ["drug", "dose", "unit", "frequency", "conditions", "y_dose", "y_unit", "y_freq", "y_cond", "y_comb"]
EV_KEYS = ["ev_dosage", "ev_warnings", "ev_contra"]
problems = []
for split, df in RAW.items():
    miss = [f"{k} -> '{COLS[k]}'" for k in REQUIRED if COLS.get(k) is None or COLS[k] not in df.columns]
    if miss: problems.append(f"[{split}] missing required columns (config key -> expected name):\n    " + "\n    ".join(miss)
                             + f"\n    available columns: {list(df.columns)}")
if problems: _fail("Column mapping does not match the data:\n" + "\n".join(problems) + "\nFix the COLS dict in section 2.")

ROW_LEVEL_EVIDENCE = all(COLS.get(k) in df.columns for df in RAW.values() for k in EV_KEYS)
EVID_MAP = {}
if not _unset(EVIDENCE_PATH) and Path(EVIDENCE_PATH).exists():
    ev_df = read_table(EVIDENCE_PATH)
    miss = [f"{k} -> '{v}'" for k, v in EVID_COLS.items() if v not in ev_df.columns]
    if miss: _fail("EVIDENCE_PATH is missing columns:\n    " + "\n    ".join(miss) + f"\n    available: {list(ev_df.columns)}")
    for r in ev_df.to_dict("records"):
        k = str(r[EVID_COLS["drug"]]).strip().lower()
        e = EVID_MAP.setdefault(k, dict(dosage=[], warnings=[], contra=[]))
        for src, dst in [("dosage", "dosage"), ("warnings", "warnings"), ("contraindications", "contra")]:
            v = r[EVID_COLS[src]]
            if isinstance(v, str) and v.strip(): e[dst].append(v.strip())
    EVID_MAP = {k: {a: "\n".join(b) for a, b in v.items()} for k, v in EVID_MAP.items()}
elif not ROW_LEVEL_EVIDENCE:
    _fail("No evidence available: the example files lack evidence columns "
          f"({[COLS.get(k) for k in EV_KEYS]}) and EVIDENCE_PATH={EVIDENCE_PATH!r} is unset/missing.")
print(f"Evidence source: {'row-level columns' if ROW_LEVEL_EVIDENCE else 'per-drug lookup'}"
      f"{' (+ per-drug fallback)' if ROW_LEVEL_EVIDENCE and EVID_MAP else ''} | drugs in evidence file: {len(EVID_MAP)}")
for k in ["group_id", "error_type"]:
    if COLS.get(k) is None or any(COLS[k] not in df.columns for df in RAW.values()):
        print(f"⚠️  optional column '{k}' not found -> " + ("leakage check cannot be verified!" if k == "group_id" else "error-type breakdown will be 'unknown'"))

# --- normalisation helpers ---
_LBL = {"SAFE": {"SAFE", "OK", "NONE", "NO", "NORMAL", "FALSE", "0", "CORRECT", "NO_ERROR", "NOERROR", "PASS", "0.0"},
        "WARNING": {"WARNING", "WARN", "MODERATE", "MEDIUM", "TRUE", "1", "YES", "ERROR", "ERR", "1.0"},
        "HIGH": {"HIGH", "SEVERE", "CRITICAL", "2", "DANGER", "DANGEROUS", "UNSAFE", "2.0"}}
_UNMAPPED = Counter()
def norm_label(x):
    if x is None or (isinstance(x, float) and math.isnan(x)): return None
    if isinstance(x, (bool, np.bool_)): return "WARNING" if x else "SAFE"
    s = str(x).strip().upper()
    for k, vals in _LBL.items():
        if s in vals: return k
    _UNMAPPED[s] += 1
    return None

def parse_conditions(x):
    if x is None or (isinstance(x, float) and math.isnan(x)): return []
    if isinstance(x, (list, tuple, np.ndarray)): items = [str(i) for i in x]
    else:
        s = str(x).strip()
        if s.startswith("["):
            try: items = [str(i) for i in json.loads(s)]
            except Exception: items = re.split(CONDITION_SEPARATORS, s.strip("[]"))
        else: items = re.split(CONDITION_SEPARATORS, s)
    items = [i.strip().strip("'\"") for i in items]
    return [i for i in items if i and i.lower() not in {"none", "n/a", "nan", "na", "no known conditions", "null"}]

def _s(x):
    return "" if x is None or (isinstance(x, float) and math.isnan(x)) else str(x)

def lookup_evidence(drug):
    k = str(drug).strip().lower()
    if k in EVID_MAP: return EVID_MAP[k]
    for kk, v in EVID_MAP.items():
        if len(kk) >= 4 and (kk in k or k in kk): return v
    return None

def build_records(df, split):
    recs = []
    for i, r in enumerate(df.to_dict("records")):
        g = lambda key: r.get(COLS[key]) if COLS.get(key) else None
        ev = dict(dosage=_s(g("ev_dosage")), warnings=_s(g("ev_warnings")), contra=_s(g("ev_contra")))
        if not any(v.strip() for v in ev.values()):
            m = lookup_evidence(g("drug"))
            if m: ev = dict(m)
        rng_ = None
        if COLS.get("range_min") and COLS.get("range_max") and COLS["range_min"] in df.columns and COLS["range_max"] in df.columns:
            try:
                rng_ = dict(lo=float(g("range_min")), hi=float(g("range_max")), unit=_s(g("range_unit")) or None,
                            max_daily=(float(g("max_daily")) if COLS.get("max_daily") and not pd.isna(g("max_daily")) else None))
                if any(math.isnan(rng_[k]) for k in ("lo", "hi")): rng_ = None
            except Exception: rng_ = None
        rec = dict(
            id=_s(g("example_id")) or f"{split}_{i}", split=split,
            group=_s(g("group_id")) or None,
            drug=_s(g("drug")).strip(), dose=g("dose"), unit=_s(g("unit")).strip(), freq=_s(g("frequency")).strip(),
            conditions=parse_conditions(g("conditions")),
            error_type=_s(g("error_type")) or "unknown",
            y={"dose": norm_label(g("y_dose")), "unit": norm_label(g("y_unit")), "freq": norm_label(g("y_freq")),
               "cond": norm_label(g("y_cond")), "comb": norm_label(g("y_comb"))},
            ev=ev, ev_missing=not any(v.strip() for v in ev.values()), rng=rng_,
            ref=dict(explanation=_s(g("ref_explanation")) or None, matched_warning=_s(g("ref_matched_warning")) or None,
                     evidence=_s(g("ref_evidence")) or None))
        recs.append(rec)
    ids = [r["id"] for r in recs]
    if len(set(ids)) != len(ids):                      # make ids unique so caching/joins are safe
        for j, r in enumerate(recs): r["id"] = f"{split}_{j}_{r['id']}"
    return recs

ALL_RECS = {s: build_records(df, s) for s, df in RAW.items()}
if _UNMAPPED:
    _fail(f"Unrecognised label values (add aliases to _LBL or clean the data): {dict(_UNMAPPED.most_common(10))}")

# --- leakage check: original prescriptions must not appear in more than one split ---
if all(r["group"] for rs in ALL_RECS.values() for r in rs):
    G = {s: {r["group"] for r in rs} for s, rs in ALL_RECS.items()}
    leaks = {(a, b): len(G[a] & G[b]) for a, b in [("train", "val"), ("train", "test"), ("val", "test")]}
    print("Leakage check (shared original-prescription groups):", leaks)
    if any(leaks.values()): _fail(f"DATA LEAKAGE: original prescriptions shared across splits: {leaks}")
else:
    for rs in ALL_RECS.values():
        for r in rs: r["group"] = r["group"] or r["id"]
    print("⚠️  Leakage check skipped (no group_id column).")

# --- seeded subsets (same test subset for EVERY system) ---
def subset(recs, n, seed=SEED, keep_groups=False):
    if n is None or len(recs) <= n: return list(recs)
    rng = random.Random(seed)
    if keep_groups:
        by_g = defaultdict(list)
        for r in recs: by_g[r["group"]].append(r)
        gids = sorted(by_g); rng.shuffle(gids); out = []
        for g in gids:
            if len(out) >= n: break
            out.extend(by_g[g])
        return out[:n]
    by_t = defaultdict(list)
    for r in sorted(recs, key=lambda r: r["id"]): by_t[r["error_type"]].append(r)
    for t in by_t: rng.shuffle(by_t[t])
    out, tot = [], len(recs)
    for t in sorted(by_t):
        out.extend(by_t[t][: max(1, round(n * len(by_t[t]) / tot))])
    rng.shuffle(out)
    return out[:n]

SPLIT_RECS = {"train": subset(ALL_RECS["train"], P["train_max"], keep_groups=True),
              "val":   subset(ALL_RECS["val"], P["val_dev_max"]),
              "test":  subset(ALL_RECS["test"], P["test_max"])}
VAL_LOSS_RECS = subset(ALL_RECS["val"], P["val_loss_max"], seed=SEED + 1)
SUBSET_INFO = {s: dict(used=len(SPLIT_RECS[s]), available=len(ALL_RECS[s])) for s in SPLIT_RECS}
SUBSET_INFO["val_loss"] = dict(used=len(VAL_LOSS_RECS), available=len(ALL_RECS["val"]))

rows = []
for s, rs in SPLIT_RECS.items():
    for cat in ["dose", "unit", "freq", "cond", "comb"]:
        c = Counter(r["y"][cat] for r in rs)
        rows.append(dict(split=s, category=cat, n=len(rs), SAFE=c.get("SAFE", 0), WARNING=c.get("WARNING", 0),
                         HIGH=c.get("HIGH", 0), missing=c.get(None, 0)))
dist = pd.DataFrame(rows)
display(dist.pivot_table(index="category", columns="split", values=["SAFE", "WARNING", "HIGH"], aggfunc="sum"))
print("Subset sizes used:", SUBSET_INFO)
print("Rows with NO evidence found (retrieval failures):", {s: sum(r["ev_missing"] for r in rs) for s, rs in SPLIT_RECS.items()})
print("Error types (train):", dict(Counter(r["error_type"] for r in SPLIT_RECS["train"]).most_common(12)))
for cat in ["dose", "unit", "freq", "cond", "comb"]:
    pos = sum(1 for r in SPLIT_RECS["test"] if r["y"][cat] not in (None, "SAFE"))
    if pos == 0: print(f"⚠️  test subset has NO positive '{cat}' examples -> its precision/recall/F1 are undefined (reported as NaN).")

## 5. Prompt template and JSON output schema
**One shared, versioned prompt** and **one output schema**, used identically by every model and variant (Base, Base+RAG, QLoRA+RAG). The schema is defined once and reused for the validity, missing-field and grounding checks.

* Evidence is shown as short, length-capped **numbered snippets** (`D#` = dosage, `W#` = warnings/precautions, `C#` = contraindications) taken from the retrieved openFDA text — plain per-drug lookup, no vector DB.
* The LLM may only interpret retrieved warnings/contraindications, match them to the patient's conditions and write a short explanation. It must never invent dosage ranges or contraindications.
* For the *no-retrieval* baseline the evidence block says "NONE PROVIDED" (everything else is identical).

In [ ]:
# ===== 5. PROMPT TEMPLATE + JSON SCHEMA (single source of truth) =====
STATUS_VALUES = ["SAFE", "WARNING", "HIGH"]
LEVELS = {"SAFE": 0, "WARNING": 1, "HIGH": 2}
LEVEL_NAMES = ["SAFE", "WARNING", "HIGH"]

# (field, kind, description shown to the model)
SCHEMA = [
    ("dose_status",          "status",      "SAFE|WARNING|HIGH - prescribed dose vs dosage evidence"),
    ("unit_status",          "status",      "SAFE|WARNING|HIGH - dose unit (mg/mcg/g) plausibility vs dosage evidence"),
    ("frequency_status",     "status",      "SAFE|WARNING|HIGH - frequency / daily dose vs dosage evidence"),
    ("condition_status",     "status",      "SAFE|WARNING|HIGH - label warning/contraindication that matches a patient condition"),
    ("combined_status",      "status",      "SAFE|WARNING|HIGH - raised when a prescription problem AND a condition risk are both present"),
    ("matched_warning",      "str_or_null", "short text of the single most relevant warning/contraindication snippet, or null"),
    ("evidence_quote_or_id", "str_or_null", "snippet id (e.g. W2) or an exact short quote copied from the snippets, or null"),
    ("explanation",          "str",         "max 3 short plain-English sentences that cite only the evidence"),
]
REQUIRED_FIELDS = [f for f, _, _ in SCHEMA]
SCHEMA_TEXT = "{" + ", ".join(f'"{f}": <{d}>' for f, _, d in SCHEMA) + "}"

SYSTEM_PROMPT = (
"You are Dose Whisperer, a research-prototype clinical-safety checker. You are NOT a clinician; your output is a prototype "
"safety flag, not a diagnosis or medical advice.\n"
"Task: check ONE prescription against the patient's conditions using ONLY the numbered drug-label evidence snippets provided.\n"
"Rules:\n"
"1. Never invent dosage ranges, contraindications, warnings or any other facts. If the snippets do not support a finding, use SAFE "
"for that field and set matched_warning and evidence_quote_or_id to null.\n"
"2. Statuses: SAFE (no issue found), WARNING (possible issue), HIGH (serious or clear issue).\n"
"3. dose_status, unit_status, frequency_status judge the prescribed dose, unit and frequency against the dosage evidence (D snippets). "
"condition_status judges whether a warning (W) or contraindication (C) snippet applies to one of the patient's conditions "
"(contraindication => HIGH, warning/precaution => WARNING). combined_status is raised when a prescription problem and a condition risk are both present.\n"
"4. matched_warning: the text of the most relevant W/C snippet, or null. evidence_quote_or_id: that snippet's id or an exact short quote from it, or null.\n"
"5. explanation: at most 3 short plain-English sentences that cite only the evidence.\n"
"6. Reply with ONE JSON object and nothing else.")

def _clean(s): return re.sub(r"\s+", " ", str(s or "")).strip()

_SENT = re.compile(r"(?<=[.;!?])\s+(?=[A-Z0-9(\[])")
def split_snippets(text, snippet_chars, max_snips, section_chars):
    parts = []
    for block in re.split(r"\n+", str(text or "")):
        parts.extend(_SENT.split(_clean(block)))
    out, seen, used = [], set(), 0
    for p in parts:
        p = _clean(p)
        if len(p) < 12 or p.lower() in seen: continue
        seen.add(p.lower())
        p = p[:snippet_chars].rstrip()
        if used + len(p) > section_chars and out: break
        out.append(p); used += len(p)
        if len(out) >= max_snips: break
    return out

def build_snippets(rec, scale=1.0, full=False):
    """Numbered evidence snippets [D#],[W#],[C#] with length caps (profile caps * scale). full=True disables caps."""
    k = 60.0 if full else scale
    sc, ms, sec = int(P["snippet_chars"] * k), max(1, int(round(P["max_snippets"] * k))), int(P["section_chars"] * k)
    sn = []
    for letter, key in [("D", "dosage"), ("W", "warnings"), ("C", "contra")]:
        for i, t in enumerate(split_snippets(rec["ev"][key], sc, ms, sec), 1):
            sn.append(dict(id=f"{letter}{i}", section=letter, text=t))
    return sn

def build_messages(rec, snippets, use_rag):
    dose_txt = f'{rec["dose"]} {rec["unit"]}'.strip()
    conds = "; ".join(rec["conditions"]) if rec["conditions"] else "none recorded"
    if use_rag:
        ev = "\n".join(f'[{s["id"]}] {s["text"]}' for s in snippets) if snippets else "(no label text could be retrieved for this drug)"
    else:
        ev = "NONE PROVIDED. Do not cite label text you cannot see; use null for matched_warning and evidence_quote_or_id."
    user = (f"PRESCRIPTION\n- drug: {rec['drug']}\n- dose: {dose_txt}\n- frequency: {rec['freq']}\n"
            f"PATIENT CONDITIONS: {conds}\n\nDRUG-LABEL EVIDENCE (openFDA)\n{ev}\n\n"
            f"Return JSON with exactly these keys: {SCHEMA_TEXT}")
    return SYSTEM_PROMPT, user

def render_prompt(tok, system, user, assistant=None):
    """Chat-template rendering. Falls back to merging the system text into the user turn for templates without a system role."""
    def _render(msgs):
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=(assistant is None))
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    if assistant is not None: msgs.append({"role": "assistant", "content": assistant})
    try:
        return _render(msgs)
    except Exception:
        msgs = [{"role": "user", "content": system + "\n\n" + user}] + ([{"role": "assistant", "content": assistant}] if assistant is not None else [])
        return _render(msgs)

def build_prompt(rec, use_rag, tok, max_prompt_tokens=None):
    """Returns dict(text, snippets, n_tokens, scale). Evidence caps are shrunk only if the prompt would exceed the token budget."""
    max_prompt_tokens = max_prompt_tokens or (P["max_seq_len"] - MAX_NEW_TOKENS)
    for s in (1.0, 0.75, 0.5, 0.35, 0.2):
        snips = build_snippets(rec, scale=s) if use_rag else []
        sysm, usr = build_messages(rec, snips, use_rag)
        text = render_prompt(tok, sysm, usr)
        n = len(tok(text, add_special_tokens=False)["input_ids"])
        if n <= max_prompt_tokens: break
    return dict(text=text, snippets=snips, n_tokens=n, scale=s)

# ---------- output parsing / schema validation ----------
def extract_json(text):
    """Lenient: first balanced {...} object in the text (handles code fences / chatter). Returns dict or None."""
    if not isinstance(text, str): return None
    t = re.sub(r"```(?:json)?", "", text, flags=re.I).strip()
    start = t.find("{")
    if start < 0: return None
    depth, in_str, esc = 0, False, False
    for i in range(start, len(t)):
        c = t[i]
        if in_str:
            if esc: esc = False
            elif c == "\\": esc = True
            elif c == '"': in_str = False
        else:
            if c == '"': in_str = True
            elif c == "{": depth += 1
            elif c == "}":
                depth -= 1
                if depth == 0:
                    try:
                        o = json.loads(t[start:i + 1]); return o if isinstance(o, dict) else None
                    except Exception:
                        return None
    return None

def is_strict_json(text):
    """Strict: the whole reply (after trimming code fences) is exactly one JSON object."""
    if not isinstance(text, str): return False
    t = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.I).strip()
    try: return isinstance(json.loads(t), dict)
    except Exception: return False

_NULLISH = {"", "null", "none", "n/a", "na", "nan", "not applicable"}
def validate_prediction(obj):
    """Check one parsed object against SCHEMA. Returns dict(parsed, schema_valid, missing, bad, extra, norm)."""
    res = dict(parsed=isinstance(obj, dict), schema_valid=False, missing=[], bad=[], extra=[], norm=None)
    if not res["parsed"]: return res
    norm = {}
    for f, kind, _ in SCHEMA:
        if f not in obj: res["missing"].append(f); norm[f] = None; continue
        v = obj[f]
        if kind == "status":
            ok = isinstance(v, str) and v.strip().upper() in STATUS_VALUES
            norm[f] = v.strip().upper() if ok else None
            if not ok: res["bad"].append(f)
        elif kind == "str_or_null":
            if v is not None and not isinstance(v, str): res["bad"].append(f); norm[f] = None
            else: norm[f] = None if (v is None or v.strip().lower() in _NULLISH) else v.strip()
        else:
            ok = isinstance(v, str) and v.strip() != ""
            norm[f] = v.strip() if ok else None
            if not ok: res["bad"].append(f)
    res["extra"] = [k for k in obj if k not in REQUIRED_FIELDS]
    res["schema_valid"] = not res["missing"] and not res["bad"]
    res["norm"] = norm
    return res

print("PROMPT_VERSION:", PROMPT_VERSION)
print("Output schema fields:", REQUIRED_FIELDS)

## 6. Deterministic checker and rules-only baseline
Pure Python (no LLM): parse dose / unit / frequency, convert **mg ↔ mcg ↔ g**, compute the **daily dose**, compare against the **retrieved dosage range**, and return `SAFE / WARNING / HIGH`.

**How errors are attributed (so the three dosage categories stay separate):**
* **dose_status** — number vs label single-dose range, in the label's unit. If the unit differs but converting lands inside the range, it's a legitimate alternative unit (no error).
* **unit_status** — raised when the number is plausible *in the label unit* but the stated unit makes the converted dose fall far outside the range (e.g. `500 mcg` where the label says `500–1000 mg`).
* **frequency_status** — doses/day vs the label's frequency range, plus the **daily dose** vs the label's max/day *when the single dose itself is fine* (otherwise the daily excess is a consequence of the dose error).

Thresholds live in `CHK` (section 2). Where the label text gives no usable range the checker **abstains** (`abstain_*`=True) instead of guessing.

The **condition matcher** is a transparent keyword matcher (condition → synonym stems) over the retrieved warning/contraindication snippets: contraindication hit ⇒ `HIGH`, warning/precaution hit ⇒ `WARNING`. Together with the combined-risk rule this is the **rules-only baseline**, and the numeric part is reused by the **"LLM + deterministic dose check"** merge.

In [ ]:
# ===== 6a. DOSE / UNIT / FREQUENCY PARSING AND LABEL-RANGE EXTRACTION =====
UNIT_MG = {"mg": 1.0, "mcg": 1e-3, "g": 1e3}          # conversion to mg
def worst(*s): return LEVEL_NAMES[max(LEVELS[x] for x in s)]

def norm_unit(u):
    s = str(u or "").strip().lower().replace("µ", "u").replace("μ", "u").replace(".", "")
    if s in ("mg", "milligram", "milligrams"): return "mg"
    if s in ("mcg", "ug", "microgram", "micrograms"): return "mcg"
    if s in ("g", "gm", "gram", "grams"): return "g"
    return s

def parse_dose(x):
    """-> (low, high) floats or None. '325-650' -> (325, 650)."""
    if x is None: return None
    if isinstance(x, (int, float, np.integer, np.floating)):
        return None if (isinstance(x, float) and math.isnan(x)) else (float(x), float(x))
    nums = re.findall(r"\d+(?:\.\d+)?", str(x).replace(",", ""))
    if not nums: return None
    v = [float(n) for n in nums[:2]]
    return (min(v), max(v))

def parse_freq_per_day(x):
    """-> (min_per_day, max_per_day, prn) or None when the frequency cannot be interpreted."""
    if x is None: return None
    s = str(x).lower().strip()
    if not s or s in ("nan", "none", "null"): return None
    s = re.sub(r"\b([bt])\.?\s*i\.?\s*d\b\.?", r"\1id", s)
    s = re.sub(r"\bq\.?\s*i\.?\s*d\b\.?", "qid", s)
    s = re.sub(r"\bq\.?\s*d\b\.?", "qd", s)
    prn = bool(re.search(r"\bprn\b|as needed|p\.r\.n", s))
    H = r"(?:h|hr|hrs|hours?)\b"
    m = (re.search(rf"\bq\s*(\d+(?:\.\d+)?)\s*(?:-|–|to)\s*(\d+(?:\.\d+)?)\s*{H}", s)
         or re.search(rf"every\s+(\d+(?:\.\d+)?)\s*(?:-|–|to)\s*(\d+(?:\.\d+)?)\s*{H}", s))
    if m:
        a, b = sorted((float(m.group(1)), float(m.group(2))))
        return (24 / b, 24 / a, prn) if a > 0 else None
    m = re.search(rf"\bq\s*(\d+(?:\.\d+)?)\s*{H}", s) or re.search(rf"every\s+(\d+(?:\.\d+)?)\s*{H}", s)
    if m:
        a = float(m.group(1)); return (24 / a, 24 / a, prn) if a > 0 else None
    m = re.search(r"(\d+)\s*(?:x|times?)\b", s)
    if m: n = float(m.group(1)); return (n, n, prn) if n > 0 else None
    for pat, n in [(r"\bthrice\b|\bthree\s+times?\b|\btid\b", 3), (r"\btwice\b|\btwo\s+times?\b|\bbid\b", 2),
                   (r"\bfour\s+times?\b|\bqid\b", 4), (r"\bfive\s+times?\b", 5), (r"\bsix\s+times?\b", 6),
                   (r"\bonce\b|\bone\s+time\b|\bqd\b|\bod\b|\bdaily\b|\bevery\s+day\b|\bqhs\b|\bhs\b|\bqam\b|\bqpm\b|nightly|at bedtime|at night", 1)]:
        if re.search(pat, s): return (float(n), float(n), prn)
    if re.search(r"\bweekly\b|once a week", s): return (1 / 7, 1 / 7, prn)
    return None

_NUM = r"(\d+(?:\.\d+)?)"
_UNIT = r"(mg|mcg|ug|g)\b"
RANGE_RE = re.compile(rf"{_NUM}\s*(?:{_UNIT})?\s*(?:to|-|–|—|and|through)\s*{_NUM}\s*{_UNIT}")
MAXD_RE = re.compile(rf"(?:max(?:imum)?(?:\s+(?:daily|total))?(?:\s+dos(?:e|age))?|not\s+(?:to\s+)?exceed|no\s+more\s+than|up\s+to)"
                     rf"[^.;]{{0,60}}?{_NUM}\s*{_UNIT}\s*(?:/|per|a|in|each|every)?\s*(?:day|24\s*-?\s*hours?|24h|daily)")
SINGLE_RE = re.compile(rf"{_NUM}\s*{_UNIT}")
_REF_CACHE = {}

def label_freq_range(t):
    vals = []
    for m in re.finditer(r"every\s+(\d+(?:\.\d+)?)\s*(?:to|-|–)\s*(\d+(?:\.\d+)?)\s*hours?", t):
        a, b = sorted((float(m.group(1)), float(m.group(2))))
        if a > 0: vals += [24 / b, 24 / a]
    for m in re.finditer(r"every\s+(\d+(?:\.\d+)?)\s*hours?", t):
        a = float(m.group(1))
        if a > 0: vals.append(24 / a)
    for pat, n in [(r"\bonce\s+(?:a\s+|per\s+)?dail?y?\b|\bonce\s+a\s+day\b|\bonce\s+daily\b", 1), (r"\btwice\s+(?:a\s+)?(?:daily|day)\b|\btwo\s+times\s+(?:a\s+)?(?:daily|day)\b", 2),
                   (r"\bthree\s+times\s+(?:a\s+)?(?:daily|day)\b", 3), (r"\bfour\s+times\s+(?:a\s+)?(?:daily|day)\b", 4)]:
        if re.search(pat, t): vals.append(float(n))
    return (min(vals), max(vals)) if vals else None

def extract_label_ref(text):
    """Heuristic extraction of the label's single-dose range / max daily dose / frequency range from retrieved dosage text."""
    key = hash(text)
    if key in _REF_CACHE: return _REF_CACHE[key]
    out = dict(lo=None, hi=None, unit=None, max_daily=None, max_daily_derived=False, freq=None, low_conf=False, source="regex")
    t = re.sub(r"(?<=\d),(?=\d{3}\b)", "", str(text or "")).lower().replace("µ", "u").replace("μ", "u")
    t = re.sub(r"\s+", " ", t)
    if not t.strip(): _REF_CACHE[key] = out; return out
    singles, dailies = [], []
    for m in RANGE_RE.finditer(t):
        lo, u1, hi, u2 = float(m.group(1)), m.group(2), float(m.group(3)), m.group(4)
        if lo > hi: continue
        tail = t[m.end(): m.end() + 40]
        if re.match(r"\s*(?:/|per)\s*(?:kg|m2|m\^2|lb)", tail): continue                 # weight/BSA-based: not comparable
        unit = norm_unit(u2)
        if u1 and norm_unit(u1) in UNIT_MG and unit in UNIT_MG and norm_unit(u1) != unit:
            lo = lo * UNIT_MG[norm_unit(u1)] / UNIT_MG[unit]
        is_daily = bool(re.match(r"\s*(?:/|per|a|each|in|\()?\s*(?:day|daily|24)", tail)) and not re.search(r"every|each dose|per dose|single", tail[:25])
        (dailies if is_daily else singles).append((lo, hi, unit))
    md_val = None
    mm = MAXD_RE.search(t)
    if mm: md_val = (float(mm.group(1)), norm_unit(mm.group(2)))
    if singles:
        u0 = singles[0][2]
        conv = lambda v, u: v * UNIT_MG[u] / UNIT_MG[u0] if (u in UNIT_MG and u0 in UNIT_MG) else v
        out.update(lo=min(conv(a, u) for a, b, u in singles), hi=max(conv(b, u) for a, b, u in singles), unit=u0)
    else:
        vals = []
        for sent in re.split(r"(?<=[.;])\s+", t):
            if not re.search(r"dos|starting|initial|usual|recommended|take|maintenance", sent): continue
            for m in SINGLE_RE.finditer(sent):
                v, u = float(m.group(1)), norm_unit(m.group(2))
                tail = sent[m.end(): m.end() + 25]
                if re.match(r"\s*(?:/|per)\s*(?:kg|m2|day|24)", tail) or re.match(r"\s*(?:in|a)\s*(?:day|24)", tail): continue
                if md_val and abs(v * UNIT_MG.get(u, 1) - md_val[0] * UNIT_MG.get(md_val[1], 1)) < 1e-9: continue
                vals.append((v, u))
        if vals:
            u0 = vals[0][1]
            vv = [v * UNIT_MG[u] / UNIT_MG[u0] for v, u in vals if u in UNIT_MG and u0 in UNIT_MG]
            if vv: out.update(lo=min(vv), hi=max(vv), unit=u0, low_conf=True)
    if md_val is None and dailies: md_val = (dailies[0][1], dailies[0][2])
    out["freq"] = label_freq_range(t)
    ref_u = out["unit"] or (md_val[1] if md_val else None)
    if md_val:
        v, u = md_val
        out["max_daily"] = v * UNIT_MG[u] / UNIT_MG[ref_u] if (u in UNIT_MG and ref_u in UNIT_MG) else v
        out["unit"] = ref_u
    elif out["hi"] is not None and out["freq"]:
        out["max_daily"], out["max_daily_derived"] = out["hi"] * out["freq"][1], True
    _REF_CACHE[key] = out
    return out

def label_ref_for(rec):
    ref = extract_label_ref(rec["ev"]["dosage"])
    if rec.get("rng"):                                   # precomputed ranges from Notebook 2 take priority
        r = rec["rng"]; ref = dict(ref)
        ref.update(lo=r["lo"], hi=r["hi"], unit=norm_unit(r["unit"]) if r.get("unit") else (ref["unit"] or norm_unit(rec["unit"])),
                   low_conf=False, source="precomputed")
        if r.get("max_daily") is not None: ref["max_daily"], ref["max_daily_derived"] = r["max_daily"], False
    return ref

In [ ]:
# ===== 6b. DETERMINISTIC DOSE CHECKER, CONDITION MATCHER, COMBINED RULE, RULES-ONLY + MERGE =====
def _status_range(v, lo, hi):
    if v > hi * (1 + 1e-9): return "WARNING" if v / hi <= CHK["over_high"] * (1 + 1e-9) else "HIGH"
    if v < lo * (1 - 1e-9):
        r = lo / v if v > 0 else float("inf")
        return "WARNING" if r < CHK["under_high"] * (1 - 1e-9) else "HIGH"
    return "SAFE"

def _status_over(x, lim):
    if x <= lim * (1 + 1e-9): return "SAFE"
    return "WARNING" if x / lim <= CHK["daily_high"] * (1 + 1e-9) else "HIGH"

def check_prescription(rec):
    """Deterministic verdict for dose / unit / frequency. Returns statuses, abstain flags and short human-readable notes."""
    ref = label_ref_for(rec)
    out = dict(dose_status="SAFE", unit_status="SAFE", frequency_status="SAFE", abstain_dose=True, abstain_freq=True,
               dose_note="", unit_note="", freq_note="", ref=ref, daily_dose=None)
    dpar, unit, fp = parse_dose(rec["dose"]), norm_unit(rec["unit"]), parse_freq_per_day(rec["freq"])
    v_raw = dpar[1] if dpar else None
    v_eff = None
    # ---------- dose + unit ----------
    if ref["lo"] is not None and ref["hi"] is not None:
        lo, hi, ru = ref["lo"], ref["hi"], ref["unit"]
        if ref["low_conf"]: lo = lo / CHK["single_ref_lo_tol"]
        out["abstain_dose"] = False
        rng_txt = f"label range {ref['lo']:g}-{ref['hi']:g} {ru}"
        if v_raw is None:
            out["dose_status"], out["dose_note"] = "WARNING", f"dose '{rec['dose']}' could not be parsed"
        elif not unit:
            out["dose_status"], out["unit_status"] = _status_range(v_raw, lo, hi), "WARNING"
            out["unit_note"] = "unit missing"; out["dose_note"] = f"{v_raw:g} (no unit) vs {rng_txt}"; v_eff = v_raw
        elif unit == ru:
            out["dose_status"] = _status_range(v_raw, lo, hi); v_eff = v_raw
            out["dose_note"] = f"{v_raw:g} {unit} vs {rng_txt}"
        elif unit in UNIT_MG and ru in UNIT_MG:
            conv = v_raw * UNIT_MG[unit] / UNIT_MG[ru]
            conv_st = _status_range(conv, lo, hi)
            if conv_st == "SAFE":
                v_eff = conv; out["dose_note"] = f"{v_raw:g} {unit} = {conv:g} {ru}, inside {rng_txt}"
            elif _status_range(v_raw, lo, hi) == "SAFE":       # number fits the label unit => the UNIT is the likely error
                gap = max(conv / hi, lo / conv) if conv > 0 else float("inf")
                out["unit_status"] = "HIGH" if gap >= CHK["unit_high"] else "WARNING"
                out["unit_note"] = f"unit '{unit}' converts {v_raw:g} {unit} to {conv:g} {ru}, far from {rng_txt}"
                v_eff = v_raw
            else:
                out["dose_status"] = conv_st; v_eff = conv
                out["dose_note"] = f"{v_raw:g} {unit} = {conv:g} {ru} vs {rng_txt}"
        else:
            out["unit_status"], out["unit_note"] = "WARNING", f"unit '{rec['unit']}' is not comparable with label unit '{ru}'"
    # ---------- frequency + daily dose ----------
    fr, md_ = ref.get("freq"), ref.get("max_daily")
    if fp is None:
        out["abstain_freq"] = False
        out["frequency_status"], out["freq_note"] = "WARNING", f"frequency '{rec['freq']}' could not be interpreted"
    elif fr is not None or md_ is not None:
        out["abstain_freq"] = False
        per_day = fp[1]; st_f = "SAFE"; notes = []
        if fr is not None:
            if per_day > fr[1] * (1 + 1e-9):
                st_f = _status_over(per_day, fr[1]); notes.append(f"{per_day:g}/day exceeds label max {fr[1]:g}/day")
            elif per_day < fr[0] * (1 - 1e-9) and fr[0] / per_day >= 2.0:
                st_f = worst(st_f, "WARNING"); notes.append(f"{per_day:g}/day is well below label {fr[0]:g}/day")
        if md_ is not None and out["dose_status"] == "SAFE" and v_eff is not None:
            daily = v_eff * per_day; out["daily_dose"] = daily
            st_d = _status_over(daily, md_)
            if st_d != "SAFE": notes.append(f"daily dose {daily:g} {ref['unit']} exceeds label max {md_:g} {ref['unit']}/day")
            st_f = worst(st_f, st_d)
        out["frequency_status"], out["freq_note"] = st_f, "; ".join(notes)
    return out

def combine_risk(dose_risk, cond_risk, mode=None):
    mode = mode or COMBINED_MODE
    if mode == "overall_max": return worst(dose_risk, cond_risk)
    if dose_risk != "SAFE" and cond_risk != "SAFE":
        return LEVEL_NAMES[min(2, max(LEVELS[dose_risk], LEVELS[cond_risk]) + 1)]
    return "SAFE"

COND_SYN = {
 "renal": ["renal", "kidney", "nephro", "creatinine", "egfr", "dialysis", "ckd"],
 "hepatic": ["hepatic", "liver", "cirrhosis", "hepatitis", "hepatotox"],
 "cardiac": ["heart failure", "cardiac", "congestive", "chf", "cardiovascular", "myocardial", "coronary artery disease", "cardiomyopathy"],
 "hypertension": ["hypertension", "hypertensive", "blood pressure"],
 "diabetes": ["diabet", "hypoglyc", "hyperglyc"],
 "respiratory": ["asthma", "bronch", "copd", "pulmonary", "respiratory"],
 "gi_ulcer": ["ulcer", "gastrointestinal", "peptic", "gi bleed"],
 "bleeding": ["bleeding", "hemorrhag", "haemorrhag", "coagulopathy", "anticoag", "thrombocytopen"],
 "pregnancy": ["pregnan", "fetal", "fetus", "lactation", "nursing mother", "breast-feed", "breastfeed"],
 "elderly": ["elderly", "geriatric", "older adult", "aged 65"],
 "seizure": ["seizure", "epilep", "convuls"],
 "thyroid": ["thyroid"],
 "allergy": ["allerg", "hypersens", "anaphyla", "angioedema"],
 "arrhythmia": ["arrhythm", "qt prolong", "fibrillation", "bradycardia", "tachycardia"],
 "depression": ["depress", "suicid"],
 "electrolyte": ["hypokalemia", "hyperkalemia", "hyponatremia", "electrolyte"],
}
_NEG = re.compile(r"\bnot contraindicated\b|\bno contraindications?\b|\bno known contraindications?\b")

def match_condition_rules(rec, snippets):
    """Keyword matcher: patient condition -> synonym stems -> W/C snippet. Returns dict(level, id, text, condition) or None."""
    best = None
    for cond in rec["conditions"]:
        cl = cond.lower()
        stems = set()
        for canon, words in COND_SYN.items():
            if any(w in cl for w in words): stems.update(words)
        if len(cl) >= 6: stems.add(cl)
        if not stems: continue
        for sn in snippets:
            if sn["section"] not in ("C", "W"): continue
            tl = sn["text"].lower()
            if _NEG.search(tl): continue
            if any(s in tl for s in stems):
                lvl = 2 if sn["section"] == "C" else 1
                if best is None or lvl > best["lvl"]:
                    best = dict(lvl=lvl, level=LEVEL_NAMES[lvl], id=sn["id"], text=sn["text"], condition=cond)
    return best

def _short(t, n=140): t = _clean(t); return t if len(t) <= n else t[:n].rsplit(" ", 1)[0]

def template_explanation(rec, det, m, labels):
    """Template ('silver') explanation built from labels + retrieved evidence. Used for training targets and as the text-metric
    reference when the dataset has no gold explanation. NOTE: text metrics against it measure style adherence, not clinical truth."""
    parts = []
    notes = [n for n in (det["dose_note"] if labels["dose"] != "SAFE" else "", det["unit_note"] if labels["unit"] != "SAFE" else "",
                         det["freq_note"] if labels["freq"] != "SAFE" else "") if n]
    if notes: parts.append("Prescription issue: " + "; ".join(notes) + ".")
    elif any(labels[k] != "SAFE" for k in ("dose", "unit", "freq")):
        parts.append("The prescribed dose, unit or frequency deviates from the label dosing evidence.")
    else: parts.append("Dose, unit and frequency are consistent with the label evidence.")
    if labels["cond"] != "SAFE" and m:
        parts.append(f"Patient condition '{m['condition']}' relates to label {'contraindication' if m['id'].startswith('C') else 'warning'} {m['id']}: \"{_short(m['text'])}\".")
    elif labels["cond"] != "SAFE":
        parts.append("A condition-related risk is expected but no matching label text was found in the evidence.")
    else: parts.append("No label warning matched the patient's conditions.")
    if labels["comb"] != "SAFE": parts.append("Both a prescription problem and a condition risk are present, so the overall risk is raised.")
    return " ".join(parts)[:600]

def build_gold(rec, snippets):
    """Gold JSON target + reference text for one example (statuses from labels; evidence/explanation from data or silver rules)."""
    y = {k: (v or "SAFE") for k, v in rec["y"].items()}
    det = check_prescription(rec)
    m = match_condition_rules(rec, snippets) if y["cond"] != "SAFE" else None
    ref = rec["ref"]
    matched = ref["matched_warning"] or (m["text"] if m else None)
    ev_id = ref["evidence"] or (m["id"] if m else None)
    expl = ref["explanation"] or template_explanation(rec, det, m, y)
    return dict(dose_status=y["dose"], unit_status=y["unit"], frequency_status=y["freq"], condition_status=y["cond"],
                combined_status=y["comb"], matched_warning=matched, evidence_quote_or_id=ev_id, explanation=expl)

def rules_predict(rec):
    """Rules-only baseline: deterministic checker + keyword condition matcher + combined rule. Returns a schema-conformant dict."""
    snips = build_snippets(rec)
    det = check_prescription(rec)
    m = match_condition_rules(rec, snips)
    cond = m["level"] if m else "SAFE"
    dose_risk = worst(det["dose_status"], det["unit_status"], det["frequency_status"])
    comb = combine_risk(dose_risk, cond)
    notes = [n for n in (det["dose_note"], det["unit_note"], det["freq_note"]) if n and (dose_risk != "SAFE")]
    expl = ("Rules check: " + ("; ".join(notes) if notes else "no dosage issue found") + ". "
            + (f"Condition '{m['condition']}' matched label snippet {m['id']}." if m else "No label snippet matched the patient's conditions."))
    return dict(dose_status=det["dose_status"], unit_status=det["unit_status"], frequency_status=det["frequency_status"],
                condition_status=cond, combined_status=comb, matched_warning=(m["text"] if m else None),
                evidence_quote_or_id=(m["id"] if m else None), explanation=expl[:600]), snips

def merge_det(pred_norm, rec):
    """'LLM + deterministic dose check': numeric verdicts (dose/unit/frequency) from Python, condition + explanation from the LLM;
    combined risk recomputed. Where the checker abstains the LLM's verdict is kept. pred_norm=None (invalid JSON) => condition SAFE."""
    base = dict(pred_norm) if pred_norm else dict(dose_status="SAFE", unit_status="SAFE", frequency_status="SAFE",
                                                  condition_status="SAFE", combined_status="SAFE", matched_warning=None,
                                                  evidence_quote_or_id=None, explanation="")
    det = check_prescription(rec)
    for k, ab in (("dose_status", "abstain_dose"), ("unit_status", "abstain_dose"), ("frequency_status", "abstain_freq")):
        if not det[ab]: base[k] = det[k]
        elif base.get(k) is None: base[k] = "SAFE"
    cond = base.get("condition_status") or "SAFE"
    base["condition_status"] = cond
    base["combined_status"] = combine_risk(worst(base["dose_status"], base["unit_status"], base["frequency_status"]), cond)
    return base

In [ ]:
# ===== 6c. SELF-TEST of the deterministic components (must pass before any GPU time is spent) =====
def _mk(drug, dose, unit, freq, conds=()):
    sp = DEMO_DRUGS[drug]
    return dict(id="t", drug=drug, dose=dose, unit=unit, freq=freq, conditions=list(conds), error_type="t", y={}, rng=None,
                ev=dict(dosage=sp["dosage"], warnings=sp["warnings"], contra=sp["contra"]), ref={})
def _st(rec): d = check_prescription(rec); return d["dose_status"], d["unit_status"], d["frequency_status"]

assert extract_label_ref(DEMO_DRUGS["acetaminophen"]["dosage"])["hi"] == 1000
assert parse_freq_per_day("q6h")[:2] == (4.0, 4.0) and parse_freq_per_day("BID")[0] == 2 and parse_freq_per_day("every 4 to 6 hours")[:2] == (4.0, 6.0)
assert parse_freq_per_day("twice daily")[0] == 2 and parse_freq_per_day("garbage??") is None
assert _st(_mk("acetaminophen", 500, "mg", "q6h")) == ("SAFE", "SAFE", "SAFE")
assert _st(_mk("acetaminophen", 5000, "mg", "q6h"))[0] == "HIGH"
assert _st(_mk("acetaminophen", 250, "mg", "q6h"))[0] == "WARNING"
assert _st(_mk("acetaminophen", 0.5, "g", "q6h")) == ("SAFE", "SAFE", "SAFE")             # legitimate alternative unit
assert _st(_mk("acetaminophen", 500, "mcg", "q6h")) == ("SAFE", "HIGH", "SAFE")           # unit error, not a dose error
assert _st(_mk("acetaminophen", 500, "mg", "q1h"))[2] == "HIGH"
assert _st(_mk("digoxin", 250, "mcg", "qd")) == ("SAFE", "SAFE", "SAFE")
assert _st(_mk("digoxin", 250, "mg", "qd"))[1] == "HIGH"
assert _st(_mk("metformin", 500, "mg", "bid")) == ("SAFE", "SAFE", "SAFE")
_r = rules_predict(_mk("metformin", 500, "mg", "bid", ["renal impairment"]))[0]
assert _r["condition_status"] == "HIGH" and _r["evidence_quote_or_id"].startswith("C")
_r = rules_predict(_mk("ibuprofen", 400, "mg", "q6h", ["heart failure"]))[0]
assert _r["condition_status"] == "WARNING" and _r["combined_status"] == "SAFE"      # both_required mode
if COMBINED_MODE == "both_required":
    assert combine_risk("HIGH", "WARNING") == "HIGH" and combine_risk("WARNING", "WARNING") == "HIGH" and combine_risk("HIGH", "SAFE") == "SAFE"
assert _r["explanation"] and rules_predict(_mk("lisinopril", 100, "mg", "qd", ["pregnancy"]))[0]["combined_status"] == "HIGH"
_m = merge_det(dict(dose_status="SAFE", unit_status="SAFE", frequency_status="SAFE", condition_status="WARNING", combined_status="SAFE",
                    matched_warning=None, evidence_quote_or_id=None, explanation="x"), _mk("acetaminophen", 5000, "mg", "q6h"))
assert _m["dose_status"] == "HIGH" and _m["combined_status"] == "HIGH"
assert validate_prediction(extract_json('Sure! ```json\n{"dose_status":"safe","unit_status":"SAFE","frequency_status":"SAFE","condition_status":"SAFE","combined_status":"SAFE","matched_warning":null,"evidence_quote_or_id":null,"explanation":"ok"}\n```'))["schema_valid"]
assert not validate_prediction(extract_json('{"dose_status":"MAYBE"}'))["schema_valid"] and extract_json("no json here") is None
print("✅ deterministic self-test passed")

### 6d. Diagnostic: rules-only *frequency* checker vs. the frequency labels (validation only)
In the first smoke test the rules-only frequency check had 10 false positives and 2 misses on 24 rows. The labels come from 02b (`FREQ_RULES`), while this checker also flags *total daily dose > label max/day*. This cell cross-tabulates the two on the **validation** split (the test split stays locked) and lists disagreements, so you can decide whether the label definition or the checker needs aligning before the full run. It changes nothing.

In [ ]:
# ===== 6d. DIAGNOSTIC (validation only): rules-only frequency status vs frequency label =====
try:
    _rows = []
    for _r in SPLIT_RECS["val"]:
        _d = check_prescription(_r)
        _rows.append(dict(id=_r["id"], drug=_r["drug"], dose=_r["dose"], unit=_r["unit"], freq=_r["freq"], error_type=_r.get("error_type"),
                          label=(_r["y"].get("freq") or "SAFE"), rules=_d["frequency_status"], note=_d["freq_note"]))
    _df = pd.DataFrame(_rows)
    print(f"n={len(_df)} validation rows")
    display(pd.crosstab(_df["label"], _df["rules"], rownames=["label (02b)"], colnames=["rules-only"]))
    _bad = _df[_df["label"] != _df["rules"]]
    print(f"disagreements: {len(_bad)} of {len(_df)}")
    print("by error_type:", _bad["error_type"].value_counts().to_dict())
    display(_bad.head(12))
except Exception as _e:
    print("diagnostic skipped:", repr(_e))


## 7. Model loader (4-bit NF4) and one-example sanity check
* **Base and QLoRA both run in 4-bit** (NF4, double quantisation, bf16/fp16 compute) so the comparison isolates *fine-tuning*. An optional bf16 (unquantised) base row is added as a reference on the A100 profile.
* **MedGemma-4B-it is multimodal** — it is loaded with its image-text class (`AutoModelForImageTextToText`, falling back to `AutoModelForCausalLM`) but used **text-only**: only `input_ids` are passed and chat-template messages are plain strings. LoRA adapters are attached **only to the language-model layers** (regex target), not the vision tower; training uses `token_type_ids=0` (all text) and eager attention as recommended for Gemma-3.
* A sanity check per model verifies loading, the chat template and JSON output on one validation example before any long run.

In [ ]:
# ===== 7a. VARIANTS, QUANTISATION, LOADERS, LOW-LEVEL GENERATION =====
VARIANTS = {
    "base_norag":    dict(label="Base (no retrieval)",            rag=False, adapter=False, quant="4bit"),
    "base_rag":      dict(label="Base + RAG",                     rag=True,  adapter=False, quant="4bit"),
    "qlora_rag":     dict(label="QLoRA + RAG",                    rag=True,  adapter=True,  quant="4bit"),
    "base_bf16_rag": dict(label="Base bf16 + RAG (reference)",    rag=True,  adapter=False, quant="bf16"),
}
DERIVED = {"qlora_rag_det": dict(label="QLoRA + RAG + deterministic dose check", source="qlora_rag")}
VARIANT_LABEL = {**{k: v["label"] for k, v in VARIANTS.items()}, **{k: v["label"] for k, v in DERIVED.items()}, "rules_only": "Rules only (no LLM)"}
EVAL_VARIANTS = ["base_norag", "base_rag", "qlora_rag"] + (["base_bf16_rag"] if (RUN_BF16_REFERENCE and PROFILE == "a100_80gb") else [])

_COMPUTE_DT = torch.bfloat16 if P["compute_dtype"] == "bfloat16" else torch.float16
QUANT_INFO = dict(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=P["compute_dtype"])
def make_bnb_config():
    return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=_COMPUTE_DT)

def adapter_dir(key): return OUT_DIR / "adapters" / key
def adapter_ready(key):
    d = adapter_dir(key)
    return (d / "adapter_config.json").exists() and (d / "training_info.json").exists()

def load_tokenizer(key):
    tok = AutoTokenizer.from_pretrained(MODEL_REGISTRY[key]["hf_id"], token=HF_TOKEN)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = "left"          # left padding for batched generation (training collator pads on the right itself)
    return tok

def _model_classes(key):
    if MODEL_REGISTRY[key]["family"] == "gemma3":
        return [c for c in (AutoModelForImageTextToText, AutoModelForCausalLM) if c is not None]   # multimodal class first, text-only use
    return [AutoModelForCausalLM]

def load_model(key, quant="4bit", attn=None):
    """Load a model on GPU 0. quant: '4bit' (NF4+double quant) or 'bf16'. Falls back across attention implementations / classes."""
    cfg = MODEL_REGISTRY[key]; attn = attn or P["attn"]
    kw = dict(token=HF_TOKEN, device_map={"": 0}, low_cpu_mem_usage=True, torch_dtype=_COMPUTE_DT if quant == "4bit" else torch.bfloat16)
    if quant == "4bit": kw["quantization_config"] = make_bnb_config()
    last = None
    for cls in _model_classes(key):
        for a in dict.fromkeys([attn, "sdpa", "eager"]):
            try:
                m = cls.from_pretrained(cfg["hf_id"], attn_implementation=a, **kw)
                m.eval(); print(f"  loaded {cfg['hf_id']} via {cls.__name__} | attn={a} | quant={quant}")
                return m
            except Exception as e:
                last = e; clear_gpu()
    raise RuntimeError(f"could not load {cfg['hf_id']}: {last!r}")

def attach_adapter(model, path):
    m = PeftModel.from_pretrained(model, str(path), is_trainable=False)
    m.eval(); return m

def get_stop_ids(tok, model=None):
    ids = set()
    if tok.eos_token_id is not None: ids.add(int(tok.eos_token_id))
    e = getattr(getattr(model, "generation_config", None), "eos_token_id", None)
    if isinstance(e, int): ids.add(e)
    elif e: ids.update(int(x) for x in e)
    for name in ["<end_of_turn>", "<|end|>", "<|im_end|>", "<|eot_id|>", "<|endoftext|>"]:
        tid = tok.convert_tokens_to_ids(name)
        if isinstance(tid, int) and tid >= 0 and tid != tok.unk_token_id: ids.add(tid)
    return sorted(ids)

def generate_batch(model, tok, texts, stop_ids):
    """Greedy generation for a batch of rendered prompts. Returns (decoded_texts, n_generated_tokens_per_row, wall_seconds)."""
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False)   # chat template already contains BOS
    enc.pop("token_type_ids", None)                                                 # text-only: never pass image/segment inputs
    enc = {k: v.to(model.device) for k, v in enc.items()}
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(**enc, max_new_tokens=DECODING["max_new_tokens"], do_sample=False, num_beams=1,
                             temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id, eos_token_id=stop_ids)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    new = out[:, enc["input_ids"].shape[1]:]
    stop = set(stop_ids); counts = []
    for row in new.tolist():
        n = len(row)
        for k, t in enumerate(row):
            if t in stop: n = k + 1; break
        counts.append(n)
    return tok.batch_decode(new, skip_special_tokens=True), counts, dt

In [ ]:
# ===== 7b. SANITY CHECK PER MODEL (loading + chat template + JSON output on one validation example) =====
def sanity_check(key):
    print(f"\n--- sanity check: {key} ({MODEL_REGISTRY[key]['hf_id']}) ---")
    tok = load_tokenizer(key)
    rec = SPLIT_RECS["val"][0]
    pr = build_prompt(rec, True, tok)
    print(f"prompt tokens: {pr['n_tokens']} (evidence scale {pr['scale']}) | stop ids: {get_stop_ids(tok)}")
    print("rendered prompt head:\n" + pr["text"][:350].replace("\n", "\n  ") + "\n  ...\nrendered prompt tail:\n  " + pr["text"][-250:].replace("\n", "\n  "))
    model = load_model(key, "4bit")
    try:
        texts, n, dt = generate_batch(model, tok, [pr["text"]], get_stop_ids(tok, model))
        v = validate_prediction(extract_json(texts[0]))
        print(f"generated {n[0]} tokens in {dt:.1f}s | strict JSON: {is_strict_json(texts[0])} | schema-valid: {v['schema_valid']}")
        print("raw output:\n  " + texts[0][:600].replace("\n", "\n  "))
        if not v["schema_valid"]:
            print("⚠️  Output is not schema-valid. That is a legitimate *measured* outcome for a base model (JSON validity is a reported metric), "
                  "not a loading failure; the pipeline continues.")
        print(f"peak VRAM so far: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
    finally:
        del model; clear_gpu()
    return True

SANITY_OK = {}
for _k in list(ACTIVE_MODELS):
    if is_done(f"sanity_{_k}"): SANITY_OK[_k] = True; print(f"sanity check for {_k}: cached ✅"); continue
    ok = safe_run(f"sanity:{_k}", sanity_check, _k)
    SANITY_OK[_k] = bool(ok)
    if ok: mark_done(f"sanity_{_k}")
    else: ACTIVE_MODELS.remove(_k); print(f"⛔ {_k} removed from this run (see {FAIL_LOG})")
print("Models continuing:", ACTIVE_MODELS)

## 8. Inference function with batching, checkpointing and resume
`run_inference` generates greedily in length-sorted batches (OOM-safe: halves the batch size and retries), **appends every batch to a JSONL file** (raw output + the exact evidence snippets used + token counts + latency) and writes a metadata file with peak GPU memory, latency and tokens/second. Re-running skips finished variants and **resumes partial files**. Predictions are parsed/scored later, so metrics can be recomputed without regenerating.

In [ ]:
# ===== 8. BATCHED INFERENCE WITH CHECKPOINTING =====
def pred_path(split, key, variant): return OUT_DIR / "preds" / f"{split}__{key}__{variant}.jsonl"
def meta_path(split, key, variant): return OUT_DIR / "preds" / f"{split}__{key}__{variant}.meta.json"

def read_jsonl(p):
    rows = []
    if Path(p).exists():
        with open(p) as f:
            for line in f:
                try: rows.append(json.loads(line))
                except Exception: pass        # tolerate a half-written last line after a crash
    return rows

def read_meta(split, key, variant):
    try: return json.loads(meta_path(split, key, variant).read_text())
    except Exception: return {}

def variant_done(split, key, variant):
    m = read_meta(split, key, variant)
    if not m.get("completed"): return False
    if FORCE_RERUN and split != "test": return False
    got = {r["id"] for r in read_jsonl(pred_path(split, key, variant))}
    return {r["id"] for r in SPLIT_RECS[split]} <= got

_CLEARED = set()
def run_inference(model, tok, key, variant, split):
    if split == "test" and not TEST_UNLOCKED:
        raise RuntimeError("Test split is locked until section 11b (it must be touched once, after all training is finished).")
    recs, vcfg = SPLIT_RECS[split], VARIANTS[variant]
    pp, mp = pred_path(split, key, variant), meta_path(split, key, variant)
    if FORCE_RERUN and split != "test" and pp not in _CLEARED:
        pp.unlink(missing_ok=True); mp.unlink(missing_ok=True); _CLEARED.add(pp)
    done = {r["id"] for r in read_jsonl(pp)}
    todo = [r for r in recs if r["id"] not in done]
    prev = read_meta(split, key, variant)
    if not todo and prev.get("completed"):
        print(f"  [{split}] {key}/{variant}: cached ({len(done)} predictions)"); return prev
    if split == "test":
        with open(OUT_DIR / "logs" / "test_access_log.jsonl", "a") as f:
            f.write(json.dumps(dict(time=now_iso(), model=key, variant=variant, n_todo=len(todo))) + "\n")
    stop_ids = get_stop_ids(tok, model)
    prompts = [build_prompt(r, vcfg["rag"], tok) for r in tqdm(todo, desc="building prompts", leave=False)]
    order = sorted(range(len(todo)), key=lambda i: -prompts[i]["n_tokens"])     # longest first => OOM surfaces early
    bs = P["gen_bs"]; i = 0
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    pbar = tqdm(total=len(todo), desc=f"{split}:{key}:{variant}")
    with open(pp, "a") as f:
        while i < len(order):
            idx = order[i:i + bs]
            try:
                texts, ntok, dt = generate_batch(model, tok, [prompts[j]["text"] for j in idx], stop_ids)
            except torch.cuda.OutOfMemoryError:
                clear_gpu()
                if bs == 1: raise
                bs = max(1, bs // 2); print(f"  OOM -> batch size {bs}"); continue
            for j, t, n in zip(idx, texts, ntok):
                f.write(json.dumps(dict(id=todo[j]["id"], model=key, variant=variant, split=split, raw_output=t,
                                        n_prompt_tokens=prompts[j]["n_tokens"], n_gen_tokens=n, latency_s=dt / len(idx),
                                        batch_size=len(idx), evidence_scale=prompts[j]["scale"],
                                        evidence_used=[[s["id"], s["text"]] for s in prompts[j]["snippets"]])) + "\n")
            f.flush(); i += len(idx); pbar.update(len(idx))
    pbar.close()
    rows = read_jsonl(pp)
    peak = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    meta = dict(prev); meta.update(
        model=key, hf_id=MODEL_REGISTRY[key]["hf_id"], variant=variant, split=split, n=len(rows), completed=True,
        peak_vram_batched_gb=max(peak, prev.get("peak_vram_batched_gb", 0.0)), gen_batch_size=P["gen_bs"],
        mean_latency_batched_s=float(np.mean([r["latency_s"] for r in rows])) if rows else float("nan"),
        tokens_per_s_batched=float(sum(r["n_gen_tokens"] for r in rows) / max(1e-9, sum(r["latency_s"] for r in rows))),
        mean_prompt_tokens=float(np.mean([r["n_prompt_tokens"] for r in rows])) if rows else float("nan"),
        profile=PROFILE, prompt_version=PROMPT_VERSION, decoding=DECODING, finished_at=now_iso())
    mp.write_text(json.dumps(meta, indent=1))
    return meta

def measure_single_latency(model, tok, key, variant, n=8):
    """bs=1 latency / tokens-per-second / peak memory on validation examples (deployment-relevant numbers)."""
    stop_ids = get_stop_ids(tok, model); lat, toks = [], 0
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    for k, r in enumerate(SPLIT_RECS["val"][:max(2, n)]):
        pr = build_prompt(r, VARIANTS[variant]["rag"], tok)
        _, nt, dt = generate_batch(model, tok, [pr["text"]], stop_ids)
        if k > 0: lat.append(dt); toks += nt[0]            # first call = warm-up
    return dict(single_latency_s=float(np.mean(lat)), single_tokens_per_s=float(toks / max(1e-9, sum(lat))),
                peak_vram_single_gb=(torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0))

def _eval_one(model, tok, key, variant, split):
    meta = run_inference(model, tok, key, variant, split)
    if split == "val" and "single_latency_s" not in read_meta(split, key, variant):
        m = read_meta(split, key, variant); m.update(measure_single_latency(model, tok, key, variant))
        meta_path(split, key, variant).write_text(json.dumps(m, indent=1))
    return meta

def eval_model(key, split, variants):
    """Evaluate the requested variants of one model on one split. Loads each needed model once (base variants first, adapter last)."""
    todo = [v for v in variants if not variant_done(split, key, v)]
    if not todo: print(f"[{key}:{split}] all variants cached ✅"); return
    tok = load_tokenizer(key)
    t4 = sorted([v for v in todo if VARIANTS[v]["quant"] == "4bit"], key=lambda v: VARIANTS[v]["adapter"])
    if t4:
        model = load_model(key, "4bit")
        try:
            for v in t4:
                if VARIANTS[v]["adapter"]:
                    if not adapter_ready(key): print(f"  [{key}] no adapter available -> skipping {v}"); continue
                    model = attach_adapter(model, adapter_dir(key))     # peft wraps in place => base variants were run first
                safe_run(f"eval:{key}:{v}:{split}", _eval_one, model, tok, key, v, split)
        finally:
            del model; clear_gpu()
    for v in [v for v in todo if VARIANTS[v]["quant"] == "bf16"]:
        model = load_model(key, "bf16")
        try: safe_run(f"eval:{key}:{v}:{split}", _eval_one, model, tok, key, v, split)
        finally: del model; clear_gpu()

## 9. Base and Base + RAG evaluation (validation / dev split)
Runs the **Base (no retrieval)** and **Base + RAG** variants (and the optional bf16 reference) of each model on the **validation subset**. These dev runs let you look at behaviour, JSON validity and latency *before* any training decision; the **test split stays locked**. Each model is wrapped in `try/except`; finished variants are cached.

In [ ]:
# ===== 9. BASE / BASE+RAG on VALIDATION (dev) =====
DEV_BASE_VARIANTS = [v for v in EVAL_VARIANTS if not VARIANTS[v]["adapter"]]
for key in ACTIVE_MODELS:
    print(f"\n######## {key}: base variants on VAL ########")
    safe_run(f"dev_base:{key}", eval_model, key, "val", DEV_BASE_VARIANTS)

# quick dev peek: JSON validity of what has been generated so far
for key in ACTIVE_MODELS:
    for v in DEV_BASE_VARIANTS:
        rows = read_jsonl(pred_path("val", key, v))
        if rows:
            ok = np.mean([validate_prediction(extract_json(r["raw_output"]))["schema_valid"] for r in rows])
            m = read_meta("val", key, v)
            print(f"{key:9s} {v:14s} n={len(rows):4d} schema-valid JSON={ok:.1%} | single-example latency={m.get('single_latency_s', float('nan')):.2f}s")

## 10. QLoRA training per model
**Train split only. Validation is used only for early stopping (validation loss) and dev evaluation; hyper-parameters were fixed a-priori** (conservative: ~2 epochs, LR 1e-4, LoRA r=16/α=32/dropout 0.05 on all attention + MLP linear layers; for MedGemma only on the language model).

* Loss is computed **on the JSON answer only** (prompt tokens masked), with the **RAG prompt** (QLoRA + RAG is the variant under study).
* Targets are the gold labels + matched evidence + explanation (from your data if mapped, else template "silver" text — see section 0, assumption 5).
* **Adapters only** are saved, **immediately** after each model finishes, together with a `training_info.json` (hyper-parameters, quantisation config, time, GPU/VRAM, train/val loss). A finished adapter is never retrained on re-run; interrupted runs resume from the last checkpoint.
* `low_vram_6gb`: batch 1 + gradient accumulation, gradient checkpointing, paged 8-bit optimizer; MedGemma QLoRA is skipped (inference-only).

In [ ]:
# ===== 10a. SFT DATA PREPARATION (completion-only loss) =====
class ListDataset(torch.utils.data.Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def prepare_sft_items(tok, recs, max_len, desc):
    items, skipped = [], 0
    for r in tqdm(recs, desc=desc, leave=False):
        pr = build_prompt(r, True, tok, max_prompt_tokens=max(256, max_len - 400))    # keep room for the answer
        gold_txt = json.dumps(build_gold(r, pr["snippets"]), ensure_ascii=False)
        sysm, usr = build_messages(r, pr["snippets"], True)
        try: full = render_prompt(tok, sysm, usr, assistant=gold_txt)
        except Exception: full = None
        target = full[len(pr["text"]):] if (full and full.startswith(pr["text"])) else gold_txt + (tok.eos_token or "")
        p_ids = tok(pr["text"], add_special_tokens=False)["input_ids"]
        t_ids = tok(target, add_special_tokens=False)["input_ids"]
        if len(p_ids) + len(t_ids) > max_len: skipped += 1; continue                # never truncate the answer
        items.append(dict(input_ids=p_ids + t_ids, labels=[-100] * len(p_ids) + t_ids))
    if skipped: print(f"  skipped {skipped} over-long examples")
    return items

def make_collator(pad_id, needs_tti):
    def collate(batch):
        L = max(len(b["input_ids"]) for b in batch)
        ids = torch.full((len(batch), L), pad_id, dtype=torch.long)
        lab = torch.full((len(batch), L), -100, dtype=torch.long)
        att = torch.zeros((len(batch), L), dtype=torch.long)
        for i, b in enumerate(batch):
            n = len(b["input_ids"])
            ids[i, :n] = torch.tensor(b["input_ids"]); lab[i, :n] = torch.tensor(b["labels"]); att[i, :n] = 1
        out = dict(input_ids=ids, attention_mask=att, labels=lab)
        if needs_tti: out["token_type_ids"] = torch.zeros_like(ids)      # Gemma-3 / MedGemma: all tokens are text
        return out
    return collate

In [ ]:
# ===== 10b. QLoRA TRAINING (one model) =====
def plot_training_curves(key, log_history):
    tr = [(h["step"], h["loss"]) for h in log_history if "loss" in h]
    ev = [(h["step"], h["eval_loss"]) for h in log_history if "eval_loss" in h]
    if not tr and not ev: return
    plt.figure(figsize=(6, 3.5))
    if tr: plt.plot(*zip(*tr), label="train loss", color="#1f77b4")
    if ev: plt.plot(*zip(*ev), label="validation loss", color="#d95f02", marker="o")
    plt.xlabel("optimizer step"); plt.ylabel("loss"); plt.title(f"QLoRA training - {MODEL_NAME[key]}"); plt.legend(); plt.tight_layout()
    plt.savefig(OUT_DIR / "figs" / f"training_curves_{key}.png", dpi=150); plt.close()

def train_qlora(key):
    cfg, ad = MODEL_REGISTRY[key], adapter_dir(key)
    if adapter_ready(key) and not FORCE_RERUN:
        print(f"[{key}] adapter already trained -> skipping"); return
    ad.mkdir(parents=True, exist_ok=True)
    if key in P["skip_qlora"]:
        info = dict(model=key, hf_id=cfg["hf_id"], status="skipped_inference_only", reason=f"profile {PROFILE}", profile=PROFILE)
        (ad / "training_info.json").write_text(json.dumps(info, indent=1)); print(f"[{key}] QLoRA skipped on {PROFILE} (inference-only)"); return
    set_seed_all()
    tok = load_tokenizer(key); max_len = P["max_seq_len"]
    train_items = prepare_sft_items(tok, SPLIT_RECS["train"], max_len, "tokenising train")
    val_items = prepare_sft_items(tok, VAL_LOSS_RECS, max_len, "tokenising val")
    print(f"[{key}] train examples: {len(train_items)} | val-loss examples: {len(val_items)} | max_len {max_len}")
    model = load_model(key, "4bit", attn=cfg["train_attn"])
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=P["grad_ckpt"],
                                            gradient_checkpointing_kwargs={"use_reentrant": False})
    lcfg = LoraConfig(r=TRAIN_HP["lora_r"], lora_alpha=TRAIN_HP["lora_alpha"], lora_dropout=TRAIN_HP["lora_dropout"],
                      bias="none", task_type="CAUSAL_LM", target_modules=cfg["lora_targets"])
    model = get_peft_model(model, lcfg)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad); n_all = sum(p.numel() for p in model.parameters())
    print(f"[{key}] trainable params: {n_tr / 1e6:.1f}M of {n_all / 1e6:.0f}M ({100 * n_tr / n_all:.2f}%)")

    eff = P["train_bs"] * P["grad_accum"]
    steps_per_epoch = max(1, math.ceil(len(train_items) / eff))
    total_steps = steps_per_epoch * TRAIN_HP["epochs"]
    eval_steps = max(2, min(total_steps, steps_per_epoch // TRAIN_HP["evals_per_epoch"] or 2))
    # NOTE: transformers>=5 removed TrainingArguments(warmup_ratio=...); an explicit integer warmup_steps works in v4 and v5.
    sig = inspect.signature(TrainingArguments.__init__).parameters
    strat = {"eval_strategy": "steps"} if "eval_strategy" in sig else {"evaluation_strategy": "steps"}
    use_bf16 = P["compute_dtype"] == "bfloat16"
    ckpt = OUT_DIR / "ckpt" / key
    args = TrainingArguments(
        output_dir=str(ckpt), per_device_train_batch_size=P["train_bs"], per_device_eval_batch_size=max(1, P["train_bs"]),
        gradient_accumulation_steps=P["grad_accum"], learning_rate=TRAIN_HP["learning_rate"], num_train_epochs=TRAIN_HP["epochs"],
        lr_scheduler_type=TRAIN_HP["lr_scheduler"], warmup_steps=max(1, int(round(TRAIN_HP["warmup_ratio"] * total_steps))), weight_decay=TRAIN_HP["weight_decay"],
        max_grad_norm=TRAIN_HP["max_grad_norm"], logging_steps=max(1, eval_steps // 2), eval_steps=eval_steps,
        save_strategy="steps", save_steps=eval_steps, save_total_limit=2, load_best_model_at_end=True,
        metric_for_best_model="eval_loss", greater_is_better=False, bf16=use_bf16, fp16=not use_bf16, optim=P["optim"],
        gradient_checkpointing=P["grad_ckpt"], gradient_checkpointing_kwargs={"use_reentrant": False} if P["grad_ckpt"] else None,
        report_to="none", remove_unused_columns=False, label_names=["labels"], seed=SEED, data_seed=SEED,
        dataloader_num_workers=2, **strat)
    trainer = Trainer(model=model, args=args, train_dataset=ListDataset(train_items), eval_dataset=ListDataset(val_items),
                      data_collator=make_collator(tok.pad_token_id, cfg["needs_token_type_ids"]),
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=TRAIN_HP["early_stopping_patience"])])
    last = get_last_checkpoint(str(ckpt)) if ckpt.exists() else None
    if last: print(f"[{key}] resuming from {last}")
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    res = trainer.train(resume_from_checkpoint=last)
    minutes = (time.time() - t0) / 60
    peak = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    st = trainer.state
    trainer.model.save_pretrained(str(ad))                                   # adapters ONLY, saved immediately
    tr_losses = [h["loss"] for h in st.log_history if "loss" in h]
    ev_losses = [(h["step"], h["eval_loss"]) for h in st.log_history if "eval_loss" in h]
    info = dict(model=key, hf_id=cfg["hf_id"], status="trained", profile=PROFILE, quick_mode=QUICK_MODE, gpu=GPU_NAME,
                learning_rate=TRAIN_HP["learning_rate"], epochs_planned=TRAIN_HP["epochs"], epochs_completed=round(float(st.epoch or 0), 2),
                global_steps=int(st.global_step), max_steps=int(st.max_steps), early_stopped=bool(st.global_step < st.max_steps),
                per_device_batch=P["train_bs"], grad_accum=P["grad_accum"], effective_batch=eff,
                lora_r=TRAIN_HP["lora_r"], lora_alpha=TRAIN_HP["lora_alpha"], lora_dropout=TRAIN_HP["lora_dropout"],
                target_modules=str(cfg["lora_targets"]), quantization=QUANT_INFO, optimizer=P["optim"], max_seq_len=max_len,
                gradient_checkpointing=P["grad_ckpt"], attn_implementation=cfg["train_attn"], n_train=len(train_items), n_val_loss=len(val_items),
                train_time_min=round(minutes, 2), peak_vram_gb=round(peak, 2),
                final_train_loss=(round(float(np.mean(tr_losses[-3:])), 4) if tr_losses else None), train_loss_avg=round(float(res.training_loss), 4),
                best_val_loss=(round(float(st.best_metric), 4) if st.best_metric is not None else (min(v for _, v in ev_losses) if ev_losses else None)),
                final_val_loss=(round(ev_losses[-1][1], 4) if ev_losses else None), seed=SEED,
                trainable_params=int(n_tr), total_params=int(n_all), adapter_path=str(ad), finished_at=now_iso())
    (ad / "training_info.json").write_text(json.dumps(info, indent=1))
    plot_training_curves(key, st.log_history)
    print(f"[{key}] ✅ trained in {minutes:.1f} min | best val loss {info['best_val_loss']} | peak VRAM {peak:.1f} GB | adapter -> {ad}")
    shutil.rmtree(ckpt, ignore_errors=True)
    del trainer, model; clear_gpu()

for key in ACTIVE_MODELS:
    print(f"\n######## {key}: QLoRA training ########")
    safe_run(f"train:{key}", train_qlora, key)

def load_training_info():
    rows = []
    for k in MODEL_REGISTRY:
        p = adapter_dir(k) / "training_info.json"
        if p.exists():
            try: rows.append(json.loads(p.read_text()))
            except Exception: pass
    return pd.DataFrame(rows)
TRAIN_INFO = load_training_info()
if len(TRAIN_INFO):
    display(TRAIN_INFO.drop(columns=[c for c in ["quantization", "adapter_path"] if c in TRAIN_INFO.columns]))
    TRAIN_INFO.to_csv(OUT_DIR / "tables" / "training_info.csv", index=False)

## 11. QLoRA + RAG evaluation (validation / dev split)
Reloads each base model in 4-bit, attaches the **saved adapter** (this also proves the saved artifact works) and generates on the validation subset with the identical prompt, schema and decoding settings.

In [ ]:
# ===== 11. QLoRA + RAG on VALIDATION (dev) =====
for key in ACTIVE_MODELS:
    print(f"\n######## {key}: QLoRA+RAG on VAL ########")
    safe_run(f"dev_qlora:{key}", eval_model, key, "val", ["qlora_rag"])
for key in ACTIVE_MODELS:
    rows = read_jsonl(pred_path("val", key, "qlora_rag"))
    if rows:
        ok = np.mean([validate_prediction(extract_json(r["raw_output"]))["schema_valid"] for r in rows])
        print(f"{key:9s} qlora_rag     n={len(rows):4d} schema-valid JSON={ok:.1%}")

## 11b. 🔒 Final one-time TEST pass
Training and all dev decisions are finished, so the **test split is unlocked now** and every LLM variant is generated **once** on the **same test subset**. Predictions are cached on disk; an audit trail is written to `logs/test_access_log.jsonl`. Re-running this cell does **not** regenerate finished test predictions (and `FORCE_RERUN` is ignored for test). If you change anything about training or prompts afterwards, treat the test results as no longer clean.

In [ ]:
# ===== 11b. UNLOCK TEST + FINAL PASS =====
TEST_UNLOCKED = True
print("🔓 TEST split unlocked (one pass; predictions are cached and never regenerated).")
for key in ACTIVE_MODELS:
    print(f"\n######## {key}: TEST pass ({', '.join(EVAL_VARIANTS)}) ########")
    safe_run(f"test:{key}", eval_model, key, "test", EVAL_VARIANTS)
print("\nTest predictions available:", sorted(p.name for p in (OUT_DIR / "preds").glob("test__*.jsonl")))

## 12. Ablation systems (same test set for all)
Five system-level variants are assembled from the cached predictions — **no extra generation is needed**, and every one is scored on the **identical test subset**:

| # | System | Where it comes from |
|---|---|---|
| 1 | **Rules only** | deterministic checker + keyword condition matcher + combined rule (no LLM) |
| 2 | **Base LLM** | base model, 4-bit, no retrieval |
| 3 | **LLM + RAG** | base model, 4-bit, retrieved openFDA snippets in the prompt |
| 4 | **Fine-tuned LLM + RAG** | QLoRA adapter + retrieved snippets |
| 5 | **Fine-tuned LLM + RAG + deterministic dose checking** | numeric verdicts (dose/unit/frequency) from Python, condition matching + explanation from the fine-tuned LLM, combined risk recomputed |

In [ ]:
# ===== 12. SYSTEM / ABLATION ASSEMBLY =====
ABLATION_MAP = [("Rules only", "rules_only"), ("Base LLM", "base_norag"), ("LLM + RAG", "base_rag"),
                ("Fine-tuned LLM + RAG", "qlora_rag"), ("Fine-tuned LLM + RAG + deterministic dose check", "qlora_rag_det")]
CAT_FIELDS = [("dose", "dose_status"), ("unit", "unit_status"), ("freq", "frequency_status"),
              ("cond", "condition_status"), ("comb", "combined_status")]
CAT_NAME = {"dose": "Dose errors", "unit": "Unit errors", "freq": "Frequency errors", "cond": "Condition risks", "comb": "Combined risks"}

_PRED_CACHE = {}
def _empty_pred(rec):
    return dict(id=rec["id"], present=False, parsed=False, schema_valid=False, strict=False, missing=[], bad=[], extra=[], norm=None,
                evidence_used=[], raw="", n_gen_tokens=0, latency_s=float("nan"))

def load_llm_preds(split, key, variant):
    ck = (split, key, variant)
    if ck in _PRED_CACHE: return _PRED_CACHE[ck]
    rows = {r["id"]: r for r in read_jsonl(pred_path(split, key, variant))}
    out = []
    for rec in SPLIT_RECS[split]:
        r = rows.get(rec["id"])
        if r is None: out.append(_empty_pred(rec)); continue
        v = validate_prediction(extract_json(r["raw_output"]))
        out.append(dict(id=rec["id"], present=True, parsed=v["parsed"], schema_valid=v["schema_valid"], strict=is_strict_json(r["raw_output"]),
                        missing=v["missing"], bad=v["bad"], extra=v["extra"], norm=v["norm"], evidence_used=r["evidence_used"],
                        raw=r["raw_output"], n_gen_tokens=r["n_gen_tokens"], latency_s=r["latency_s"]))
    _PRED_CACHE[ck] = out
    return out

def load_rules_preds(split):
    ck = (split, "-", "rules_only")
    if ck in _PRED_CACHE: return _PRED_CACHE[ck]
    out = []
    for rec in SPLIT_RECS[split]:
        t0 = time.perf_counter(); pred, snips = rules_predict(rec); dt = time.perf_counter() - t0
        out.append(dict(id=rec["id"], present=True, parsed=True, schema_valid=True, strict=True, missing=[], bad=[], extra=[], norm=pred,
                        evidence_used=[[s["id"], s["text"]] for s in snips], raw=json.dumps(pred), n_gen_tokens=0, latency_s=dt))
    _PRED_CACHE[ck] = out
    return out

def load_derived_preds(split, key, variant):
    ck = (split, key, variant)
    if ck in _PRED_CACHE: return _PRED_CACHE[ck]
    src = load_llm_preds(split, key, DERIVED[variant]["source"])
    out = []
    for rec, p in zip(SPLIT_RECS[split], src):
        q = dict(p)
        if p["present"]: q["norm"] = merge_det(p["norm"], rec)
        out.append(q)
    _PRED_CACHE[ck] = out
    return out

def available_systems(split):
    """All systems that can be scored on this split: rules-only + every completed LLM variant + derived variants."""
    sysl = [dict(model="-", variant="rules_only", kind="rules")]
    for key in MODEL_REGISTRY:
        for v in EVAL_VARIANTS:
            if variant_done(split, key, v): sysl.append(dict(model=key, variant=v, kind="llm"))
        if variant_done(split, key, "qlora_rag"): sysl.append(dict(model=key, variant="qlora_rag_det", kind="derived"))
    return sysl

def system_preds(split, sd):
    if sd["kind"] == "rules": return load_rules_preds(split)
    if sd["kind"] == "llm": return load_llm_preds(split, sd["model"], sd["variant"])
    return load_derived_preds(split, sd["model"], sd["variant"])

def pred_status(p, field):
    n = p.get("norm"); v = n.get(field) if n else None
    return v if v in LEVELS else "SAFE"        # invalid / missing => SAFE (counted as a miss)

SYSTEMS_DEV = available_systems("val")
print("Systems scorable on VAL :", [f"{s['model']}|{s['variant']}" for s in SYSTEMS_DEV])
SYSTEMS_TEST = available_systems("test") if TEST_UNLOCKED else []
print("Systems scorable on TEST:", [f"{s['model']}|{s['variant']}" for s in SYSTEMS_TEST])

# quick look: distribution of predicted condition/combined status per test system (full metrics follow in section 13)
rows = []
for sd in SYSTEMS_TEST:
    ps = system_preds("test", sd)
    for cat, field in CAT_FIELDS:
        c = Counter(pred_status(p, field) for p in ps)
        rows.append(dict(model=sd["model"], variant=sd["variant"], category=cat, **{k: c.get(k, 0) for k in LEVEL_NAMES}))
if rows: display(pd.DataFrame(rows).pivot_table(index=["model", "variant"], columns="category", values="HIGH", aggfunc="sum").rename_axis(columns="# predicted HIGH"))

## 13. Metrics, confusion matrices, text metrics and reliability checks
**Classification** — reported **separately** for dose, unit, frequency, condition and combined risk (positive = `WARNING`/`HIGH`): accuracy, precision, recall (= sensitivity), specificity, F1, false-positive rate, false-negative rate, a confusion matrix, 3-class accuracy, and **support counts next to every metric** (small test sets are noisy; metrics with no positives are `NaN`). Invalid/unparseable outputs count as `SAFE` (a miss). The headline "Precision / Recall / F1" in the comparison table are the **macro-average over the five categories** that have positives in the test subset.

**Explanation text** — BLEU, ROUGE-1/2/L, METEOR, BERTScore-F1 against the reference explanation (gold if mapped, else template "silver" text — see section 0). These are *style-similarity* scores and are **not** treated as correctness measures.

**Reliability** — JSON validity (strict / lenient parse / schema-valid), missing fields, **evidence grounding** (cited snippet id or quote really is in the retrieved text), **unsupported claims** (condition flagged without grounded evidence), and **hallucinated medical information**: an *invented dosage range* (a dose/amount in the explanation or matched warning that appears neither in the evidence nor in the prescription nor as a computed daily dose) or an *invented contraindication* (claims "contraindicated" / cites a warning that is not in the retrieved text).

**System** — latency per example (single-request and batched), tokens/second, peak GPU memory.

In [ ]:
# ===== 13a. METRIC FUNCTIONS =====
BERTSCORE_MODEL = "roberta-large"      # A100 has the memory; use "distilbert-base-uncased" on small GPUs
TEXT_METRICS = True
NAN = float("nan")

def binary_metrics(yt, yp):
    yt, yp = np.asarray(yt), np.asarray(yp)
    pt, pp = yt > 0, yp > 0
    tp, fp = int((pt & pp).sum()), int((~pt & pp).sum()); fn, tn = int((pt & ~pp).sum()), int((~pt & ~pp).sum())
    n, npos, nneg = len(yt), tp + fn, tn + fp
    div = lambda a, b: a / b if b > 0 else NAN
    rec, spec = div(tp, npos), div(tn, nneg)
    prec = div(tp, tp + fp) if (tp + fp) > 0 else (0.0 if npos > 0 else NAN)
    f1 = (2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0) if (npos > 0 and not math.isnan(prec)) else NAN
    return dict(n=n, support_pos=npos, support_neg=nneg, accuracy=div(tp + tn, n), precision=prec, recall=rec, sensitivity=rec,
                specificity=spec, f1=f1, fpr=div(fp, nneg), fnr=div(fn, npos), tp=tp, fp=fp, fn=fn, tn=tn,
                acc_3class=float((yt == yp).mean()) if n else NAN)

# ---------- reliability / grounding ----------
_QTY = re.compile(r"(\d+(?:\.\d+)?)\s*(mg|mcg|ug|g|ml)\b")
def _norm(s): return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9%.]+", " ", str(s).lower().replace("µ", "u").replace("μ", "u"))).strip()
def _toks(s): return {w for w in _norm(s).split() if len(w) >= 3}
def _qty(v, u):
    u = norm_unit(u)
    return ("mass", round(v * UNIT_MG[u], 5)) if u in UNIT_MG else (u, round(v, 5))
def _quantities(text):
    return {_qty(float(m.group(1)), m.group(2)) for m in _QTY.finditer(str(text).lower().replace(",", "").replace("µ", "u").replace("μ", "u"))}

def allowed_quantities(rec, ev_texts):
    allowed = set()
    for t in ev_texts: allowed |= _quantities(t)
    dp, u, fp = parse_dose(rec["dose"]), norm_unit(rec["unit"]), parse_freq_per_day(rec["freq"])
    if dp:
        for v in set(dp):
            allowed.add(_qty(v, u))
            if fp:
                for k in {fp[0], fp[1]}: allowed.add(_qty(v * k, u))
    return allowed

_ID_RE = re.compile(r"^\[?\s*([DWC]\d+)\s*\]?\s*[:\-]?\s*(.*)$", re.I | re.S)
def quote_grounded(q, ev):
    """None = nothing cited; True/False = whether the cited id / quote really exists in the evidence shown to the model."""
    if not q: return None
    ids = {i for i, _ in ev}; ev_all = " ".join(_norm(t) for _, t in ev)
    m = _ID_RE.match(q.strip()); txt = q
    if m and m.group(1).upper() in ids:
        if not m.group(2).strip(): return True
        txt = m.group(2)
    elif m and not m.group(2).strip(): return False                      # cited an id that was never shown
    frags = [f for f in (_norm(x) for x in re.split(r"…|\.\.\.", txt)) if len(f) >= 8]
    return bool(frags) and all(f in ev_all for f in frags)

def warning_grounded(w, ev):
    if not w: return None
    m = _ID_RE.match(w.strip())
    if m and m.group(1).upper() in {i for i, _ in ev} and not m.group(2).strip(): return True
    wt = _toks(m.group(2) if (m and m.group(1).upper() in {i for i, _ in ev}) else w)
    if not wt: return False
    return max([len(wt & _toks(t)) / len(wt) for _, t in ev] + [0.0]) >= 0.8

def reliability_row(rec, p):
    n, ev = p["norm"], p["evidence_used"]
    q, w, expl = n.get("evidence_quote_or_id"), n.get("matched_warning"), (n.get("explanation") or "")
    qg, wg = quote_grounded(q, ev), warning_grounded(w, ev)
    cited = (q is not None) or (w is not None)
    ungrounded = (qg is False) or (wg is False)
    cond_flag = (n.get("condition_status") or "SAFE") != "SAFE"
    unsupported = cond_flag and not ((qg is True) or (wg is True))
    ev_all = " ".join(_norm(t) for _, t in ev)
    invented_contra = ("contraindicat" in expl.lower() or "contraindicat" in (w or "").lower()) and ("contraindicat" not in ev_all)
    claimed = _quantities(expl) | _quantities(w or "")
    invented_range = bool(claimed - allowed_quantities(rec, [t for _, t in ev]))
    return dict(cited=cited, grounded=(cited and not ungrounded), ungrounded=ungrounded, unsupported_claim=unsupported,
                invented_contraindication=invented_contra, invented_range=invented_range,
                hallucination=(invented_contra or invented_range or ungrounded))

# ---------- text metrics ----------
_TS, _WARNED = {}, set()
def _warn_once(name, e):
    if name not in _WARNED: _WARNED.add(name); print(f"⚠️  {name} unavailable -> NaN ({e!r})")

def text_metrics(hyps, refs):
    out = dict(n_text=len(hyps), bleu=NAN, rouge1=NAN, rouge2=NAN, rougeL=NAN, meteor=NAN, bertscore_f1=NAN)
    if not hyps or not TEXT_METRICS: return out
    try:
        import sacrebleu; out["bleu"] = sacrebleu.corpus_bleu(hyps, [refs]).score / 100.0       # 0..1
    except Exception as e: _warn_once("BLEU (sacrebleu)", e)
    try:
        from rouge_score import rouge_scorer
        sc = _TS.setdefault("rouge", rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True))
        s = [sc.score(r, h) for h, r in zip(hyps, refs)]
        for k in ("rouge1", "rouge2", "rougeL"): out[k] = float(np.mean([x[k].fmeasure for x in s]))
    except Exception as e: _warn_once("ROUGE (rouge-score)", e)
    try:
        from nltk.translate.meteor_score import meteor_score
        tk = lambda s: re.findall(r"\w+", s.lower())
        out["meteor"] = float(np.mean([meteor_score([tk(r)], tk(h)) for h, r in zip(hyps, refs)]))
    except Exception as e: _warn_once("METEOR (nltk/wordnet)", e)
    try:
        from bert_score import BERTScorer
        if "bert" not in _TS:
            _TS["bert"] = BERTScorer(model_type=BERTSCORE_MODEL, lang="en", rescale_with_baseline=False,
                                     device="cuda" if torch.cuda.is_available() else "cpu")
        _, _, F = _TS["bert"].score(hyps, refs, batch_size=64)
        out["bertscore_f1"] = float(F.mean())
    except Exception as e: _warn_once("BERTScore", e)
    return out

REF_EXPL = {}
def reference_explanation(rec):
    if rec["id"] not in REF_EXPL: REF_EXPL[rec["id"]] = build_gold(rec, build_snippets(rec))["explanation"]
    return REF_EXPL[rec["id"]]

# ---------- system metrics ----------
def system_metrics(split, sd, preds):
    lat = [p["latency_s"] for p in preds if p["present"] and not math.isnan(p["latency_s"])]
    if sd["kind"] == "rules":
        l = float(np.mean(lat)) if lat else NAN
        return dict(latency_single_s=l, latency_batched_s=l, tokens_per_s_single=NAN, tokens_per_s_batched=NAN, peak_vram_single_gb=0.0, peak_vram_batched_gb=0.0)
    src = DERIVED[sd["variant"]]["source"] if sd["kind"] == "derived" else sd["variant"]
    mv, ms = read_meta("val", sd["model"], src), read_meta(split, sd["model"], src)
    return dict(latency_single_s=mv.get("single_latency_s", NAN), latency_batched_s=ms.get("mean_latency_batched_s", NAN),
                tokens_per_s_single=mv.get("single_tokens_per_s", NAN), tokens_per_s_batched=ms.get("tokens_per_s_batched", NAN),
                peak_vram_single_gb=mv.get("peak_vram_single_gb", NAN), peak_vram_batched_gb=ms.get("peak_vram_batched_gb", NAN))

def evaluate_system(split, sd):
    recs, preds = SPLIT_RECS[split], system_preds(split, sd); n = len(recs)
    cats = {}
    for cat, field in CAT_FIELDS:
        yt, yp = [], []
        for r, p in zip(recs, preds):
            t = r["y"][cat]
            if t is None: continue
            yt.append(LEVELS[t]); yp.append(LEVELS[pred_status(p, field)])
        cats[cat] = binary_metrics(yt, yp) if yt else None
    ok = [c for c in cats.values() if c and c["support_pos"] > 0]
    summ = dict(n=n, categories_in_macro=len(ok),
                precision=float(np.mean([c["precision"] for c in ok])) if ok else NAN,
                recall=float(np.mean([c["recall"] for c in ok])) if ok else NAN,
                f1=float(np.mean([c["f1"] for c in ok])) if ok else NAN,
                accuracy=float(np.mean([c["accuracy"] for c in cats.values() if c])))
    miss_by_field = Counter(f for p in preds if p["parsed"] for f in p["missing"])
    js = dict(n_predictions_present=sum(p["present"] for p in preds),
              json_strict_rate=sum(p["strict"] for p in preds) / n, json_parse_rate=sum(p["parsed"] for p in preds) / n,
              json_validity=sum(p["schema_valid"] for p in preds) / n,
              missing_field_rate=sum(1 for p in preds if p["parsed"] and p["missing"]) / n,
              bad_value_rate=sum(1 for p in preds if p["parsed"] and p["bad"]) / n, missing_by_field=dict(miss_by_field))
    rel_rows = [reliability_row(r, p) for r, p in zip(recs, preds) if p["parsed"] and p["norm"]]
    m = max(1, len(rel_rows)); cited = [x for x in rel_rows if x["cited"]]
    rel = dict(n_parsed=len(rel_rows), evidence_cited_rate=len(cited) / m,
               grounding_rate=(sum(x["grounded"] for x in cited) / len(cited)) if cited else NAN,
               unsupported_claim_rate=sum(x["unsupported_claim"] for x in rel_rows) / m,
               invented_dose_range_rate=sum(x["invented_range"] for x in rel_rows) / m,
               invented_contraindication_rate=sum(x["invented_contraindication"] for x in rel_rows) / m,
               hallucination_rate=sum(x["hallucination"] for x in rel_rows) / m)
    return dict(sd=sd, split=split, cat=cats, summary=summ, json=js, rel=rel, sys=system_metrics(split, sd, preds),
                text=dict(n_text=0, bleu=NAN, rouge1=NAN, rouge2=NAN, rougeL=NAN, meteor=NAN, bertscore_f1=NAN))

def short_name(sd): return "Rules only" if sd["kind"] == "rules" else f"{MODEL_NAME[sd['model']]}\n{VARIANT_LABEL[sd['variant']]}"

def plot_confusions(evals, tag):
    paths = []
    for cat, _ in CAT_FIELDS:
        items = [(k, e) for k, e in evals.items() if e["cat"].get(cat)]
        if not items: continue
        ncol = min(4, len(items)); nrow = math.ceil(len(items) / ncol)
        fig, axes = plt.subplots(nrow, ncol, figsize=(3.3 * ncol, 3.2 * nrow), squeeze=False)
        for ax in axes.ravel(): ax.axis("off")
        for ax, (k, e) in zip(axes.ravel(), items):
            c = e["cat"][cat]; cm = np.array([[c["tn"], c["fp"]], [c["fn"], c["tp"]]]); ax.axis("on")
            ax.imshow(cm, cmap="Blues", vmin=0)
            for (i, j), v in np.ndenumerate(cm): ax.text(j, i, str(v), ha="center", va="center", color="white" if v > cm.max() / 2 else "black", fontsize=11)
            ax.set_xticks([0, 1]); ax.set_xticklabels(["SAFE", "FLAG"], fontsize=8); ax.set_yticks([0, 1]); ax.set_yticklabels(["SAFE", "FLAG"], fontsize=8)
            ax.set_xlabel("predicted", fontsize=8); ax.set_ylabel("true", fontsize=8)
            ax.set_title(short_name(e["sd"]) + f"\n(pos={c['support_pos']}, neg={c['support_neg']})", fontsize=7)
        fig.suptitle(f"{CAT_NAME[cat]} - confusion matrices [{tag}]  (FLAG = WARNING or HIGH)", fontsize=10)
        fig.tight_layout(); p = OUT_DIR / "figs" / f"confusion_{cat}_{tag}.png"; fig.savefig(p, dpi=150); plt.close(fig); paths.append(p)
    return paths

In [ ]:
# ===== 13b. COMPUTE ALL METRICS (test = reported results; val = dev) =====
EVAL_TEST = {(sd["model"], sd["variant"]): evaluate_system("test", sd) for sd in tqdm(SYSTEMS_TEST, desc="classification+reliability (test)")}
EVAL_DEV  = {(sd["model"], sd["variant"]): evaluate_system("val", sd) for sd in tqdm(SYSTEMS_DEV, desc="classification+reliability (val/dev)")}

# --- text metrics (LLM variants only; rules-only text is templated and is NOT scored; derived reuses its source LLM's explanations) ---
def explanation_pairs(split, sd):
    hyps, refs = [], []
    for r, p in zip(SPLIT_RECS[split], system_preds(split, sd)):
        if p["parsed"] and p["norm"] and p["norm"].get("explanation"): hyps.append(p["norm"]["explanation"]); refs.append(reference_explanation(r))
    return hyps, refs

for k, e in tqdm(EVAL_TEST.items(), desc="text metrics (test)"):
    if e["sd"]["kind"] == "llm": e["text"] = text_metrics(*explanation_pairs("test", e["sd"]))
for k, e in EVAL_TEST.items():
    if e["sd"]["kind"] == "derived":
        src = EVAL_TEST.get((k[0], DERIVED[k[1]]["source"]))
        if src: e["text"] = dict(src["text"])
if "bert" in _TS: del _TS["bert"]; clear_gpu()

# --- long metrics table (one row per system x category, with support counts) ---
def metrics_long(evals, split):
    rows = []
    for (model, variant), e in evals.items():
        for cat, c in e["cat"].items():
            if c: rows.append(dict(split=split, model=MODEL_NAME.get(model, model), variant=variant, system=VARIANT_LABEL[variant],
                                   category=CAT_NAME[cat], **c))
    return pd.DataFrame(rows)
METRICS_LONG = pd.concat([metrics_long(EVAL_TEST, "test"), metrics_long(EVAL_DEV, "val")], ignore_index=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.float_format", lambda x: f"{x:.3f}")
print("Per-category classification metrics on TEST (support counts shown next to every metric):")
display(METRICS_LONG[METRICS_LONG.split == "test"][["model", "system", "category", "support_pos", "support_neg", "accuracy", "precision", "recall",
                                                   "specificity", "f1", "fpr", "fnr", "tp", "fp", "fn", "tn", "acc_3class"]])

# --- confusion-matrix images ---
CM_PATHS = plot_confusions(EVAL_TEST, "test") + plot_confusions(EVAL_DEV, "val")
print("Saved confusion matrices:", [p.name for p in CM_PATHS])
for p in [q for q in CM_PATHS if q.name.endswith("test.png")][:2]:
    display(Image(filename=str(p)))

# --- reliability + text + system tables ---
def wide_row(model, variant, e):
    s, j, r, t, y = e["summary"], e["json"], e["rel"], e["text"], e["sys"]
    row = dict(Model=MODEL_NAME.get(model, "-" if model == "-" else model), Variant=VARIANT_LABEL[variant], variant_key=variant, model_key=model, n=s["n"],
               Precision=s["precision"], Recall=s["recall"], F1=s["f1"], Accuracy=s["accuracy"], categories_in_macro=s["categories_in_macro"])
    for cat, c in e["cat"].items():
        if c:
            row.update({f"{cat}_P": c["precision"], f"{cat}_R": c["recall"], f"{cat}_F1": c["f1"], f"{cat}_spec": c["specificity"],
                        f"{cat}_FPR": c["fpr"], f"{cat}_FNR": c["fnr"], f"{cat}_pos": c["support_pos"], f"{cat}_neg": c["support_neg"]})
    row.update(BLEU=t["bleu"], ROUGE_1=t["rouge1"], ROUGE_2=t["rouge2"], ROUGE_L=t["rougeL"], METEOR=t["meteor"], BERTScore_F1=t["bertscore_f1"], n_text=t["n_text"])
    row.update({"JSON validity": j["json_validity"], "JSON parse rate": j["json_parse_rate"], "JSON strict rate": j["json_strict_rate"],
                "Missing-field rate": j["missing_field_rate"], "Bad-value rate": j["bad_value_rate"]})
    row.update({"Evidence cited rate": r["evidence_cited_rate"], "Grounding rate": r["grounding_rate"], "Unsupported-claim rate": r["unsupported_claim_rate"],
                "Invented dose-range rate": r["invented_dose_range_rate"], "Invented contraindication rate": r["invented_contraindication_rate"],
                "Hallucination rate": r["hallucination_rate"]})
    row.update({"Latency (s, single)": y["latency_single_s"], "Latency (s/ex, batched)": y["latency_batched_s"], "Tokens/s (single)": y["tokens_per_s_single"],
                "Tokens/s (batched)": y["tokens_per_s_batched"], "Memory (GB, single)": y["peak_vram_single_gb"], "Memory (GB, batched)": y["peak_vram_batched_gb"]})
    return row
WIDE_TEST = pd.DataFrame([wide_row(m, v, e) for (m, v), e in EVAL_TEST.items()])
WIDE_DEV = pd.DataFrame([wide_row(m, v, e) for (m, v), e in EVAL_DEV.items()])
print("\nHeadline (TEST):")
display(WIDE_TEST[["Model", "Variant", "n", "Precision", "Recall", "F1", "BLEU", "ROUGE_L", "JSON validity", "Grounding rate", "Hallucination rate", "Latency (s, single)", "Memory (GB, single)"]])

## 14. Comparison tables, ablation, chart and model selection
* **Model comparison table** (`Model, Variant, Precision, Recall, F1, BLEU, ROUGE-L, JSON validity, Latency, Memory`) + a wider table with the full metric set.
* **Ablation table** for the five systems on the same test subset.
* **Model selection** uses *overall* results, not a single metric: a weighted score — **macro-F1 (0.45) and JSON validity (0.25) first**, then **evidence grounding (0.15)**, **latency (0.10)** and **memory (0.05)** — computed on each model's best deployable variant (fine-tuned + RAG + deterministic dose check if available). Candidates with JSON validity < 90 % are only chosen if none qualifies. 95 % bootstrap intervals for macro-F1 are shown because test subsets are small.

In [ ]:
# ===== 14a. MODEL COMPARISON + ABLATION TABLES =====
VARIANT_ORDER = ["rules_only", "base_norag", "base_rag", "base_bf16_rag", "qlora_rag", "qlora_rag_det"]
def _sort(df):
    df = df.copy(); df["_v"] = df["variant_key"].map({v: i for i, v in enumerate(VARIANT_ORDER)}); df["_m"] = df["model_key"].map(lambda m: (m != "-", m))
    return df.sort_values(["_m", "_v"]).drop(columns=["_v", "_m"]).reset_index(drop=True)

WIDE_TEST = _sort(WIDE_TEST); WIDE_DEV = _sort(WIDE_DEV)
COMPARISON = WIDE_TEST[["Model", "Variant", "Precision", "Recall", "F1", "BLEU", "ROUGE_L", "JSON validity", "Latency (s, single)", "Memory (GB, single)"]].rename(
    columns={"ROUGE_L": "ROUGE-L", "Latency (s, single)": "Latency (s)", "Memory (GB, single)": "Memory (GB)"})
print("MODEL COMPARISON (test subset; Precision/Recall/F1 = macro over the 5 categories with positives; BLEU 0-1; latency = single request, bs=1):")
display(COMPARISON)

abl_rows = []
models_present = [m for m in dict.fromkeys(k[0] for k in EVAL_TEST) if m != "-"]
for key in ([("-")] + models_present):
    for label, v in ABLATION_MAP:
        if (key == "-") != (v == "rules_only"): continue
        e = EVAL_TEST.get((key, v))
        if not e: continue
        row = dict(Model=("-" if key == "-" else MODEL_NAME[key]), System=label, Precision=e["summary"]["precision"], Recall=e["summary"]["recall"], F1=e["summary"]["f1"])
        for cat, _ in CAT_FIELDS: row[f"F1 {cat}"] = e["cat"][cat]["f1"] if e["cat"].get(cat) else NAN
        row.update({"JSON validity": e["json"]["json_validity"], "Grounding rate": e["rel"]["grounding_rate"], "Hallucination rate": e["rel"]["hallucination_rate"],
                    "Latency (s)": e["sys"]["latency_single_s"], "n": e["summary"]["n"]})
        abl_rows.append(row)
ABLATION = pd.DataFrame(abl_rows)
print("\nABLATION (same test subset for every row; 'Rules only' is model-independent):")
display(ABLATION)

In [ ]:
# ===== 14b. BOOTSTRAP CIs + MODEL SELECTION =====
def bootstrap_macro_f1(split, sd, B=300, seed=SEED):
    recs, preds = SPLIT_RECS[split], system_preds(split, sd)
    T = {c: np.array([LEVELS[r["y"][c]] if r["y"][c] is not None else -1 for r in recs]) for c, _ in CAT_FIELDS}
    Pm = {c: np.array([LEVELS[pred_status(p, f)] for p in preds]) for c, f in CAT_FIELDS}
    rng, n, vals = np.random.default_rng(seed), len(recs), []
    for _ in range(B):
        idx = rng.integers(0, n, n); f1s = []
        for c in T:
            t = T[c][idx]; keep = t >= 0
            m = binary_metrics(t[keep], Pm[c][idx][keep]) if keep.any() else None
            if m and m["support_pos"] > 0: f1s.append(m["f1"])
        if f1s: vals.append(np.mean(f1s))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))) if vals else (NAN, NAN)

PREFERRED = ["qlora_rag_det", "qlora_rag", "base_rag", "base_norag"]     # best deployable variant per model
W = dict(f1=0.45, json=0.25, grounding=0.15, latency=0.10, memory=0.05)
cands = []
for m in models_present:
    for v in PREFERRED:
        if (m, v) in EVAL_TEST: cands.append((m, v)); break
SELECTION = pd.DataFrame()
SELECTED = None
if cands:
    rows = []
    for m, v in cands:
        e = EVAL_TEST[(m, v)]; lo, hi = bootstrap_macro_f1("test", e["sd"])
        g = e["rel"]["grounding_rate"]
        rows.append(dict(model_key=m, variant_key=v, Model=MODEL_NAME[m], Variant=VARIANT_LABEL[v], F1=e["summary"]["f1"], F1_CI_low=lo, F1_CI_high=hi,
                         json_validity=e["json"]["json_validity"], grounding=(0.0 if (g is None or math.isnan(g)) else g),
                         hallucination_rate=e["rel"]["hallucination_rate"], latency_s=e["sys"]["latency_single_s"], memory_gb=e["sys"]["peak_vram_single_gb"]))
    SELECTION = pd.DataFrame(rows)
    def _inv_minmax(s):
        s = s.fillna(s.max() if s.notna().any() else 0.0)
        return pd.Series(1.0, index=s.index) if s.max() == s.min() else 1 - (s - s.min()) / (s.max() - s.min())
    SELECTION["F1"] = SELECTION["F1"].fillna(0.0)
    SELECTION["score"] = (W["f1"] * SELECTION["F1"] + W["json"] * SELECTION["json_validity"] + W["grounding"] * SELECTION["grounding"]
                          + W["latency"] * _inv_minmax(SELECTION["latency_s"]) + W["memory"] * _inv_minmax(SELECTION["memory_gb"]))
    SELECTION["qualifies_json>=0.90"] = SELECTION["json_validity"] >= 0.90
    pool = SELECTION[SELECTION["qualifies_json>=0.90"]] if SELECTION["qualifies_json>=0.90"].any() else SELECTION
    best = pool.sort_values("score", ascending=False).iloc[0]
    SELECTION = SELECTION.sort_values("score", ascending=False).reset_index(drop=True)
    SELECTED = dict(model_key=best["model_key"], variant_key=best["variant_key"])
    others = SELECTION[~((SELECTION.model_key == best["model_key"]) & (SELECTION.variant_key == best["variant_key"]))]
    reasons = [f"Selected **{best['Model']}** with variant **{best['Variant']}** (composite score {best['score']:.3f}; weights F1 {W['f1']}, JSON validity {W['json']}, grounding {W['grounding']}, latency {W['latency']}, memory {W['memory']}).",
               f"Macro-F1 {best['F1']:.3f} (95% bootstrap CI {best['F1_CI_low']:.3f}-{best['F1_CI_high']:.3f}), JSON validity {best['json_validity']:.1%}, evidence grounding {best['grounding']:.1%}, hallucination rate {best['hallucination_rate']:.1%}, single-request latency {best['latency_s']:.2f}s, peak memory {best['memory_gb']:.1f} GB."]
    if len(others):
        o = others.iloc[0]
        close = abs(best["F1"] - o["F1"]) < max(0.02, (o["F1_CI_high"] - o["F1_CI_low"]) / 2 if not math.isnan(o["F1_CI_high"]) else 0.02)
        reasons.append(f"Runner-up: {o['Model']} / {o['Variant']} (score {o['score']:.3f}, macro-F1 {o['F1']:.3f}, JSON validity {o['json_validity']:.1%}, grounding {o['grounding']:.1%}, latency {o['latency_s']:.2f}s)."
                       + (" The macro-F1 difference is within bootstrap noise for this test subset, so the choice rests mainly on JSON validity, grounding, latency and memory." if close else ""))
    if len(others) and others.iloc[0]["score"] > best["score"]:
        reasons.append("The runner-up has a higher composite score but is below the 90% JSON-validity gate, so the best candidate that passes the gate was selected.")
    if not best["qualifies_json>=0.90"]: reasons.append("No candidate reached 90% JSON validity; the best composite score was chosen anyway - treat with caution.")
    SELECTION_REASONING = " ".join(reasons)
    print("MODEL SELECTION (best deployable variant per model):")
    display(SELECTION[["Model", "Variant", "F1", "F1_CI_low", "F1_CI_high", "json_validity", "grounding", "hallucination_rate", "latency_s", "memory_gb", "score"]])
    print("\n" + SELECTION_REASONING)
else:
    SELECTION_REASONING = "No LLM system finished on the test split, so no model could be selected."
    print("⚠️ ", SELECTION_REASONING)

# ---- comparison chart ----
def plot_comparison(df, path):
    if df.empty: return
    COL = {"rules_only": "#7f7f7f", "base_norag": "#e69f00", "base_rag": "#56b4e9", "base_bf16_rag": "#009e73", "qlora_rag": "#0072b2", "qlora_rag_det": "#d55e00"}
    d = df.reset_index(drop=True); labels = [f"{r['Model']}\n{r['Variant']}" for _, r in d.iterrows()]; y = np.arange(len(d)); colors = [COL.get(v, "#444") for v in d["variant_key"]]
    fig, axes = plt.subplots(1, 4, figsize=(20, max(4, 0.55 * len(d) + 1.5)), sharey=True)
    panels = [("F1", "Macro F1 (higher = better)", None), ("JSON validity", "JSON validity", None), ("Hallucination rate", "Hallucination rate (lower = better)", None),
              ("Latency (s, single)", "Latency, single request (s)", "Memory (GB, single)")]
    for ax, (col, title, extra) in zip(axes, panels):
        vals = d[col].astype(float).fillna(0).values; ax.barh(y, vals, color=colors); ax.set_title(title, fontsize=10); ax.invert_yaxis()
        for i, v in enumerate(vals):
            txt = f"{v:.2f}" + (f"  |  {d[extra].iloc[i]:.1f} GB" if extra and not pd.isna(d[extra].iloc[i]) else "")
            ax.text(v, i, " " + txt, va="center", fontsize=8)
        ax.set_xlim(0, max(vals.max() * 1.35, 1e-6) if col.startswith("Latency") else 1.25); ax.spines[["top", "right"]].set_visible(False)
    axes[0].set_yticks(y); axes[0].set_yticklabels(labels, fontsize=8)
    fig.suptitle("Dose Whisperer - model / variant comparison (test subset; research prototype)", fontsize=12); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)
plot_comparison(WIDE_TEST, OUT_DIR / "figs" / "model_comparison.png")
display(Image(filename=str(OUT_DIR / "figs" / "model_comparison.png"))) if (OUT_DIR / "figs" / "model_comparison.png").exists() else None

## 15. Error analysis
For the **selected system** on the test subset, every example is tagged with the failure types below (an example can have several); representative samples are listed and saved, with the exact evidence the model saw so each failure can be traced:

false positives / false negatives (per category) · **parsing failures** (invalid JSON/schema) · **unit** and **frequency** errors · **condition-matching** errors · **retrieval failures** (no evidence for the drug, or the matching warning exists in the full label text but was cut by the length caps) · **LLM reasoning failures** (the matching snippet *was* in the prompt but the model got the condition wrong, or flagged a condition with no support) · **hallucinations** (invented dose range / contraindication / ungrounded citation).

In [ ]:
# ===== 15. ERROR ANALYSIS =====
def analysis_frame(split, sd):
    rows = []
    for r, p in zip(SPLIT_RECS[split], system_preds(split, sd)):
        n = p["norm"] or {}
        row = dict(id=r["id"], drug=r["drug"], dose=f'{r["dose"]} {r["unit"]}', frequency=r["freq"], conditions="; ".join(r["conditions"]),
                   error_type=r["error_type"], evidence_missing=r["ev_missing"], parsed=p["parsed"], schema_valid=p["schema_valid"])
        for cat, field in CAT_FIELDS: row[f"true_{cat}"], row[f"pred_{cat}"] = r["y"][cat], pred_status(p, field)
        row.update(matched_warning=n.get("matched_warning"), evidence_id_or_quote=n.get("evidence_quote_or_id"), explanation=n.get("explanation"),
                   raw_output=(p["raw"] or "")[:600], evidence_used=json.dumps(p["evidence_used"], ensure_ascii=False))
        used = [dict(id=i, section=i[0], text=t) for i, t in p["evidence_used"]]
        m_used, m_full = match_condition_rules(r, used), match_condition_rules(r, build_snippets(r, full=True))
        pos = lambda s: s not in (None, "SAFE")
        rel = reliability_row(r, p) if (p["parsed"] and p["norm"]) else {}
        tags = []
        for cat, _ in CAT_FIELDS:
            if r["y"][cat] is None: continue
            if not pos(r["y"][cat]) and pos(row[f"pred_{cat}"]): tags.append(f"false_positive:{cat}")
            if pos(r["y"][cat]) and not pos(row[f"pred_{cat}"]): tags.append(f"false_negative:{cat}")
        if not p["schema_valid"]: tags.append("parsing_error")
        if r["y"]["unit"] is not None and r["y"]["unit"] != row["pred_unit"]: tags.append("unit_error")
        if r["y"]["freq"] is not None and r["y"]["freq"] != row["pred_freq"]: tags.append("frequency_error")
        if r["y"]["cond"] is not None and r["y"]["cond"] != row["pred_cond"]: tags.append("condition_matching_error")
        if r["ev_missing"] or (pos(r["y"]["cond"]) and m_full and not m_used): tags.append("retrieval_failure")
        if (pos(r["y"]["cond"]) and m_used and r["y"]["cond"] != row["pred_cond"]) or (not pos(r["y"]["cond"]) and pos(row["pred_cond"]) and not m_used):
            tags.append("llm_reasoning_failure")
        if rel.get("hallucination"): tags.append("hallucination:" + "+".join(k for k, lab in [("invented_range", "dose_range"), ("invented_contraindication", "contraindication"), ("ungrounded", "ungrounded_citation")] if rel.get(k)))
        row["tags"] = "; ".join(tags)
        rows.append(row)
    return pd.DataFrame(rows)

ERR_SAMPLES, ERR_COUNTS, ERR_FRAME = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
if SELECTED:
    sel_sd = EVAL_TEST[(SELECTED["model_key"], SELECTED["variant_key"])]["sd"]
    ERR_FRAME = analysis_frame("test", sel_sd)
    cnt = Counter(t.split(":")[0] + (":" + t.split(":")[1] if t.startswith(("false_positive", "false_negative")) else "") for tags in ERR_FRAME["tags"] for t in tags.split("; ") if t)
    ERR_COUNTS = pd.DataFrame(sorted(cnt.items()), columns=["failure_type", "n_examples"]); ERR_COUNTS["share_of_test"] = ERR_COUNTS["n_examples"] / max(1, len(ERR_FRAME))
    print(f"Error analysis for the selected system: {MODEL_NAME[SELECTED['model_key']]} / {VARIANT_LABEL[SELECTED['variant_key']]}  (n={len(ERR_FRAME)})")
    display(ERR_COUNTS)
    rng = random.Random(SEED); samples = []
    for ft in sorted(cnt):
        sub = ERR_FRAME[ERR_FRAME["tags"].apply(lambda s: any(t.startswith(ft) for t in s.split("; ")))]
        by_et = defaultdict(list)
        for _, r in sub.iterrows(): by_et[r["error_type"]].append(r)
        picks = []
        while len(picks) < min(5, len(sub)) and any(by_et.values()):          # round-robin over error types for variety
            for et in list(by_et):
                if by_et[et] and len(picks) < 5: picks.append(by_et[et].pop(rng.randrange(len(by_et[et]))))
        for r in picks: samples.append({"failure_type": ft, **r.to_dict()})
    ERR_SAMPLES = pd.DataFrame(samples)
    if len(ERR_SAMPLES):
        display(ERR_SAMPLES[["failure_type", "id", "drug", "dose", "frequency", "conditions", "error_type", "true_cond", "pred_cond", "true_dose", "pred_dose", "true_unit", "pred_unit", "true_freq", "pred_freq", "matched_warning"]].head(40))
        for ft in ["false_negative:cond", "hallucination", "retrieval_failure", "parsing_error"]:
            ex = ERR_SAMPLES[ERR_SAMPLES.failure_type == ft].head(1)
            for _, r in ex.iterrows():
                print(f"\n--- example [{ft}] id={r['id']} ---\nprescription: {r['drug']} {r['dose']} {r['frequency']} | conditions: {r['conditions']}\n"
                      f"evidence shown: {r['evidence_used'][:600]}\nmodel output: {r['raw_output'][:400]}")
else:
    print("No selected system -> error analysis skipped.")

## 16. Save all artifacts and the report fragment
Writes (all under `OUT_DIR`): `tables/model_comparison.csv`, `tables/model_comparison_wide.csv`, `tables/ablation_table.csv`, `tables/metrics.csv`, `tables/training_info.csv`, `results.json`, per-example predictions **with the evidence used** (`predictions_test_all_systems.jsonl`), `tables/error_analysis_samples.csv`, confusion-matrix and comparison images (`figs/`), `selected_model_config.json` (for the Streamlit app) and `evaluation_report_part3.md`. QLoRA adapters are already in `adapters/<model>/`.

In [ ]:
# ===== 16a. SAVE TABLES, RESULTS, PREDICTIONS =====
def clean(o):
    """JSON-safe: NaN/inf -> None, numpy -> python, DataFrame -> records."""
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)): return [clean(v) for v in o]
    if isinstance(o, pd.DataFrame): return clean(o.to_dict("records"))
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (math.isnan(float(o)) or math.isinf(float(o))) else float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, Path): return str(o)
    return o

T = OUT_DIR / "tables"
COMPARISON.to_csv(T / "model_comparison.csv", index=False)
WIDE_TEST.to_csv(T / "model_comparison_wide.csv", index=False)
WIDE_DEV.to_csv(T / "dev_val_comparison_wide.csv", index=False)
ABLATION.to_csv(T / "ablation_table.csv", index=False)
METRICS_LONG.to_csv(T / "metrics.csv", index=False); METRICS_LONG.to_csv(OUT_DIR / "metrics.csv", index=False)
if len(SELECTION): SELECTION.to_csv(T / "model_selection.csv", index=False)
if len(ERR_SAMPLES): ERR_SAMPLES.to_csv(T / "error_analysis_samples.csv", index=False)
if len(ERR_COUNTS): ERR_COUNTS.to_csv(T / "error_analysis_counts.csv", index=False)
if len(ERR_FRAME): ERR_FRAME.to_csv(T / "error_analysis_all_selected_system.csv", index=False)
if len(TRAIN_INFO): TRAIN_INFO.to_csv(T / "training_info.csv", index=False)

def system_block(e):
    return dict(system=VARIANT_LABEL[e["sd"]["variant"]], model=e["sd"]["model"], split=e["split"], summary=e["summary"],
                per_category=e["cat"], json=e["json"], reliability=e["rel"], text_metrics=e["text"], system_metrics=e["sys"])
USING_SILVER_REFS = not any(r["ref"]["explanation"] for r in SPLIT_RECS["test"])
RESULTS = dict(
    created_at=now_iso(), disclaimer="Research prototype. Outputs are prototype safety flags, not medical diagnoses or advice.",
    profile=PROFILE, quick_mode=QUICK_MODE, demo_data=USE_DEMO_DATA, seed=SEED, subset_info=SUBSET_INFO, reference_explanations="template (silver)" if USING_SILVER_REFS else "from data",
    test={f"{m}|{v}": system_block(e) for (m, v), e in EVAL_TEST.items()},
    dev_val={f"{m}|{v}": system_block(e) for (m, v), e in EVAL_DEV.items()},
    ablation=ABLATION, selection=SELECTION, selection_reasoning=SELECTION_REASONING, selected=SELECTED, training=TRAIN_INFO)
(OUT_DIR / "results.json").write_text(json.dumps(clean(RESULTS), indent=1))

# raw per-example predictions + the evidence used (test, every system) for error analysis
n_rows = 0
with open(OUT_DIR / "predictions_test_all_systems.jsonl", "w") as f:
    for (m, v), e in EVAL_TEST.items():
        for r, p in zip(SPLIT_RECS["test"], system_preds("test", e["sd"])):
            f.write(json.dumps(clean(dict(id=r["id"], model=m, variant=v, drug=r["drug"], dose=r["dose"], unit=r["unit"], frequency=r["freq"],
                    conditions=r["conditions"], error_type=r["error_type"], true=r["y"], pred=p["norm"], valid_schema=p["schema_valid"],
                    evidence_used=p["evidence_used"], raw_output=p["raw"], n_gen_tokens=p["n_gen_tokens"], latency_s=p["latency_s"],
                    reference_explanation=reference_explanation(r))), ensure_ascii=False) + "\n"); n_rows += 1
print(f"saved tables, results.json and {n_rows} per-example predictions to {OUT_DIR.resolve()}")

In [ ]:
# ===== 16b. selected_model_config.json (for the Streamlit app) =====
SELECTED_CONFIG = None
if SELECTED:
    mk, vk = SELECTED["model_key"], SELECTED["variant_key"]; cfg = MODEL_REGISTRY[mk]; e = EVAL_TEST[(mk, vk)]
    src_variant = DERIVED[vk]["source"] if vk in DERIVED else vk
    uses_adapter = VARIANTS[src_variant]["adapter"]
    sel_row = SELECTION[(SELECTION.model_key == mk) & (SELECTION.variant_key == vk)].iloc[0].to_dict()
    SELECTED_CONFIG = dict(
        disclaimer="Dose Whisperer is a research prototype. Outputs are prototype safety flags (SAFE/WARNING/HIGH), not medical diagnoses, and no substitute for clinical judgment.",
        created_at=now_iso(), selected_model=dict(key=mk, hf_id=cfg["hf_id"], family=cfg["family"], gated=cfg["gated"],
                                                  text_only=(cfg["family"] == "gemma3"), pipeline_variant=vk, llm_variant=src_variant),
        reasoning=SELECTION_REASONING,
        key_metrics=dict(macro_precision=e["summary"]["precision"], macro_recall=e["summary"]["recall"], macro_f1=e["summary"]["f1"],
                         f1_ci95=[sel_row["F1_CI_low"], sel_row["F1_CI_high"]], json_validity=e["json"]["json_validity"],
                         grounding_rate=e["rel"]["grounding_rate"], hallucination_rate=e["rel"]["hallucination_rate"],
                         latency_single_s=e["sys"]["latency_single_s"], peak_vram_single_gb=e["sys"]["peak_vram_single_gb"], n_test=e["summary"]["n"]),
        loading=dict(base_model=cfg["hf_id"], quantization=QUANT_INFO if VARIANTS[src_variant]["quant"] == "4bit" else "bf16 (no quantization)",
                     adapter_path=(str(adapter_dir(mk).resolve()) if uses_adapter else None),
                     adapter_path_relative_to_out_dir=(f"adapters/{mk}" if uses_adapter else None),
                     loading_class=("AutoModelForImageTextToText (text-only use; fallback AutoModelForCausalLM)" if cfg["family"] == "gemma3" else "AutoModelForCausalLM"),
                     attn_implementation=P["attn"], chat_template="tokenizer.apply_chat_template([system, user], add_generation_prompt=True); fall back to merging system into user",
                     tokenizer_padding_side="left", stop_token_ids_hint="eos + <end_of_turn>/<|end|>/<|im_end|>/<|eot_id|> if present"),
        prompt=dict(version=PROMPT_VERSION, system_prompt=SYSTEM_PROMPT, output_schema=[dict(field=f, kind=k, description=d) for f, k, d in SCHEMA],
                    schema_text=SCHEMA_TEXT, evidence=dict(snippet_chars=P["snippet_chars"], max_snippets_per_section=P["max_snippets"],
                    section_chars=P["section_chars"], shrink_ladder=[1.0, 0.75, 0.5, 0.35, 0.2], max_seq_len=P["max_seq_len"], snippet_ids="D#=dosage, W#=warnings, C#=contraindications")),
        decoding=dict(DECODING, seed=SEED),
        deterministic_dose_check=dict(enabled=(vk == "qlora_rag_det" or vk.endswith("_det")), thresholds=CHK, combined_mode=COMBINED_MODE,
                                      note="Python verdicts for dose/unit/frequency override the LLM where the checker does not abstain; combined risk recomputed."),
        lora=(dict(r=TRAIN_HP["lora_r"], alpha=TRAIN_HP["lora_alpha"], dropout=TRAIN_HP["lora_dropout"], target_modules=str(cfg["lora_targets"]),
                   training_info=(TRAIN_INFO[TRAIN_INFO.model == mk].iloc[0].to_dict() if len(TRAIN_INFO) and (TRAIN_INFO.model == mk).any() else None)) if uses_adapter else None),
        hardware_profile=dict(profile=PROFILE, gpu=GPU_NAME, vram_gb=round(GPU_VRAM_GB, 1), quick_mode=QUICK_MODE, subset_info=SUBSET_INFO,
                              notes="Latency/memory measured on this hardware; batched generation was used for the bulk evaluation, bs=1 for the single-request numbers."),
        ranking=SELECTION[["Model", "Variant", "F1", "json_validity", "grounding", "hallucination_rate", "latency_s", "memory_gb", "score"]],
        license_note="google/medgemma-4b-it is released under the Health AI Developer Foundations terms for research/development use." if mk == "medgemma" else None)
    (OUT_DIR / "selected_model_config.json").write_text(json.dumps(clean(SELECTED_CONFIG), indent=1))
    print("selected_model_config.json written ->", MODEL_NAME[mk], "/", VARIANT_LABEL[vk])
else:
    print("No selected model -> selected_model_config.json not written.")

In [ ]:
# ===== 16c. evaluation_report_part3.md (fragment for merging into evaluation_report.md) =====
def df_to_md(df, cols=None, floatfmt="{:.3f}"):
    if df is None or len(df) == 0: return "_(no data)_"
    d = df[cols] if cols else df
    def f(x):
        if isinstance(x, float): return "n/a" if math.isnan(x) else (f"{x:.1e}" if (x != 0 and abs(x) < 0.005) else floatfmt.format(x))
        return str(x).replace("|", "\\|").replace("\n", " ")
    head = "| " + " | ".join(map(str, d.columns)) + " |\n|" + "|".join("---" for _ in d.columns) + "|\n"
    return head + "\n".join("| " + " | ".join(f(v) for v in row) + " |" for row in d.itertuples(index=False))

lines = []
A = lines.append
A("## Part 3 - Model comparison: Base vs QLoRA (generated by Notebook 3)\n")
A("> **Research prototype.** All outputs are prototype safety flags (SAFE / WARNING / HIGH), not medical diagnoses, and not a replacement for clinical judgment.\n")
A("### 3.1 Models tested\n")
A(df_to_md(pd.DataFrame([dict(Model=MODEL_NAME[k], HF_id=MODEL_REGISTRY[k]["hf_id"], Gated=MODEL_REGISTRY[k]["gated"],
                              Use=("text-only (multimodal model)" if MODEL_REGISTRY[k]["family"] == "gemma3" else "text"),
                              Status=("QLoRA trained" if adapter_ready(k) and json.loads((adapter_dir(k) / "training_info.json").read_text()).get("status") == "trained"
                                      else "inference only / not trained")) for k in ACTIVE_MODELS])) + "\n")
A("Not used: MedGemma-27B (too large for the target hardware). Optional extras (Qwen2.5-3B-Instruct, Llama-3.2-3B-Instruct) were " + ("enabled." if (ENABLE.get("qwen") or ENABLE.get("llama")) else "off by default.") + "\n")
A("### 3.2 Prompt and output contract\n")
A(f"One versioned prompt (`{PROMPT_VERSION}`) and one strict JSON schema with fields `{', '.join(REQUIRED_FIELDS)}` were used for every model and variant. Evidence = length-capped openFDA snippets (D#/W#/C#) from a plain per-drug lookup (no vector database). Decoding: greedy (`do_sample=False`), max_new_tokens={MAX_NEW_TOKENS}, seed={SEED}. The LLM only interprets retrieved warnings/contraindications, matches them to patient conditions and writes the explanation; dose/unit/frequency arithmetic is deterministic Python.\n")
A("### 3.3 LoRA / QLoRA configuration\n")
A(f"4-bit NF4 quantisation with double quantisation (compute dtype {P['compute_dtype']}); LoRA r={TRAIN_HP['lora_r']}, alpha={TRAIN_HP['lora_alpha']}, dropout={TRAIN_HP['lora_dropout']}; learning rate {TRAIN_HP['learning_rate']}, {TRAIN_HP['epochs']} epoch(s), early stopping on validation loss (patience {TRAIN_HP['early_stopping_patience']}); loss on the JSON answer only; adapters only saved. Hardware profile: **{PROFILE}** ({GPU_NAME}).\n")
if len(TRAIN_INFO):
    A(df_to_md(TRAIN_INFO, [c for c in ["model", "status", "learning_rate", "epochs_completed", "global_steps", "early_stopped", "effective_batch", "lora_r", "lora_alpha", "lora_dropout",
                                        "train_time_min", "peak_vram_gb", "final_train_loss", "best_val_loss", "n_train", "target_modules"] if c in TRAIN_INFO.columns]) + "\n")
A("### 3.4 Evaluation setup\n")
A(f"Train/val/test were split **by original prescription** (no leakage; verified). Subsets used: `{SUBSET_INFO}`. The validation split was used only for early stopping and dev evaluation; the **test split was generated once** after training and is identical for every system. Labels come from synthetic perturbations; reference explanations are {'template-generated (silver)' if USING_SILVER_REFS else 'taken from the data'}.\n")
A("### 3.5 Model comparison (test)\n")
A(df_to_md(COMPARISON) + "\n")
A("_Precision/Recall/F1 are macro-averages over the dose, unit, frequency, condition and combined categories that have positives in the test subset; BLEU is on a 0-1 scale; latency is single-request (bs=1); memory is peak GPU allocation._\n")
A("Per-category F1 / support (test):\n")
cat_cols = ["Model", "Variant"] + [c for cat, _ in CAT_FIELDS for c in (f"{cat}_F1", f"{cat}_pos")]
A(df_to_md(WIDE_TEST, [c for c in cat_cols if c in WIDE_TEST.columns]) + "\n")
A("Explanation-text and reliability metrics (test):\n")
A(df_to_md(WIDE_TEST, ["Model", "Variant", "BLEU", "ROUGE_1", "ROUGE_2", "ROUGE_L", "METEOR", "BERTScore_F1", "JSON validity", "JSON strict rate", "Missing-field rate",
                       "Grounding rate", "Unsupported-claim rate", "Invented dose-range rate", "Invented contraindication rate", "Hallucination rate"]) + "\n")
A("BLEU/ROUGE/METEOR/BERTScore measure similarity to the reference explanation and are **not** treated as correctness measures; grounding and hallucination checks are reported alongside them.\n")
A("### 3.6 Ablation (same test subset)\n")
A(df_to_md(ABLATION) + "\n")
A("### 3.7 Model selection\n")
A(SELECTION_REASONING + "\n")
if len(SELECTION): A(df_to_md(SELECTION, ["Model", "Variant", "F1", "F1_CI_low", "F1_CI_high", "json_validity", "grounding", "hallucination_rate", "latency_s", "memory_gb", "score"]) + "\n")
A("### 3.8 Error analysis (selected system)\n")
A(df_to_md(ERR_COUNTS) + "\n")
A("Representative samples (false positives/negatives, parsing, unit, frequency, condition-matching, retrieval, LLM-reasoning and hallucination cases) are in `tables/error_analysis_samples.csv`; raw predictions with the evidence shown to the model are in `predictions_test_all_systems.jsonl`.\n")
A("### 3.9 Limitations\n")
for s in [
    "Only **Base vs QLoRA** was run (plain 16-bit LoRA is an optional extra and was not run).",
    "**MedGemma** is released for research/developer use under the Health AI Developer Foundations terms; it is not a clinical tool and these results do not validate it for patient care.",
    "**Labels come from synthetic errors** injected into MIMIC-III Demo prescriptions; the original prescriptions are not assumed to be clinically safe, and condition-risk labels depend on the Notebook-2 definitions.",
    "Results are **prototype safety flags only** - not diagnoses, not clinically validated, not a replacement for clinical judgment.",
    f"Test subset sizes are small ({SUBSET_INFO['test']['used']} examples); categories with few positives give noisy metrics (see support counts and bootstrap CIs). One seed, one run - no variance across seeds.",
    ("Reference explanations are **template-generated**, so BLEU/ROUGE/METEOR/BERTScore mostly reflect adherence to that template style." if USING_SILVER_REFS else "Reference explanations come from the dataset."),
    "The deterministic dose checker parses openFDA text with regex heuristics; where it cannot find a range it abstains, and unusual label wording can cause mis-parsed ranges. Retrieval is a plain per-drug lookup with length caps, so relevant text can be truncated.",
    "Latency and memory depend on the hardware profile used; single-request numbers (bs=1) differ from batched throughput.",
    ("**DEMO DATA was used** (synthetic, illustrative evidence text): these numbers only demonstrate that the pipeline runs." if USE_DEMO_DATA else "Evidence text and labels come from the project's own Notebook 1-2 outputs."),
    ("**QUICK_MODE** was on (tiny subsets, 1 epoch): numbers are a smoke test, not results." if QUICK_MODE else "Full-run settings were used."),
]: A(f"- {s}")
(OUT_DIR / "evaluation_report_part3.md").write_text("\n".join(lines) + "\n")
print("evaluation_report_part3.md written (", sum(len(x) for x in lines), "chars )")

## 17. Final summary

In [ ]:
# ===== 17. FINAL SUMMARY =====
MAKE_ZIP = True
print("=" * 78); print("DOSE WHISPERER - NOTEBOOK 3 SUMMARY (research prototype; flags are not diagnoses)"); print("=" * 78)
print(f"Profile: {PROFILE} | GPU: {GPU_NAME} | quick_mode={QUICK_MODE} | demo_data={USE_DEMO_DATA} | seed={SEED}")
print(f"Models run: {[MODEL_NAME[k] for k in ACTIVE_MODELS]} | test subset: {SUBSET_INFO['test']['used']} of {SUBSET_INFO['test']['available']} (same for all systems)")
if SELECTED:
    e = EVAL_TEST[(SELECTED['model_key'], SELECTED['variant_key'])]
    print(f"\nSELECTED: {MODEL_NAME[SELECTED['model_key']]} | {VARIANT_LABEL[SELECTED['variant_key']]}")
    print(f"  macro F1 {e['summary']['f1']:.3f} | precision {e['summary']['precision']:.3f} | recall {e['summary']['recall']:.3f} | JSON validity {e['json']['json_validity']:.1%} | "
          f"hallucination {e['rel']['hallucination_rate']:.1%} | latency {e['sys']['latency_single_s']:.2f}s | memory {e['sys']['peak_vram_single_gb']:.1f} GB")
    print("  " + SELECTION_REASONING.replace("**", ""))
else:
    print("\nNo model selected (see logs/failures.jsonl).")
fails = read_jsonl(FAIL_LOG)
print(f"\nLogged failures: {len(fails)}" + ("".join(f"\n  - {f['step']}: {f['error'][:140]}" for f in fails[-8:]) if fails else ""))
print("\nKey outputs:")
for p in ["tables/model_comparison.csv", "tables/model_comparison_wide.csv", "tables/ablation_table.csv", "metrics.csv", "results.json",
          "selected_model_config.json", "evaluation_report_part3.md", "predictions_test_all_systems.jsonl", "tables/error_analysis_samples.csv",
          "tables/training_info.csv", "figs/model_comparison.png"]:
    q = OUT_DIR / p; print(f"  {'OK ' if q.exists() else '-- '} {q}")
print("  adapters:", [str(adapter_dir(k)) for k in MODEL_REGISTRY if adapter_ready(k)])
print("  confusion matrices:", len(list((OUT_DIR / "figs").glob("confusion_*.png"))), "images in", OUT_DIR / "figs")
RUN_CONFIG = dict(profile=PROFILE, quick_mode=QUICK_MODE, demo_data=USE_DEMO_DATA, seed=SEED, p=P, train_hp=TRAIN_HP, decoding=DECODING, chk=CHK, combined_mode=COMBINED_MODE,
                  prompt_version=PROMPT_VERSION, cols=COLS, models={k: v["hf_id"] for k, v in MODEL_REGISTRY.items() if ENABLE.get(k)}, active=ACTIVE_MODELS,
                  subset_info=SUBSET_INFO, gpu=GPU_NAME, versions={p: _ver(p) for p in ["torch", "transformers", "peft", "bitsandbytes", "accelerate", "trl"]}, finished_at=now_iso())
(OUT_DIR / "config_snapshot.json").write_text(json.dumps(clean(RUN_CONFIG), indent=1, default=str))
if MAKE_ZIP:
    z = shutil.make_archive(str(OUT_DIR.parent / (OUT_DIR.name + "_results")), "zip", root_dir=OUT_DIR.parent, base_dir=OUT_DIR.name)
    print("\nzip of everything (incl. adapters; checkpoints were already deleted):", z)
print("\nReminder: the QLoRA adapters + selected_model_config.json are all the Streamlit app needs (plus HF access for gated base models).")
print("Reminder: this is a research prototype - prototype safety flags only, not diagnoses or clinical advice.")